# Reproducing Semenov et al. (2026) from dandiset 001517

Paper. Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler JN, Ibey BL, Kochunov P, Pakhomov AG (2026).
Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric fields.
*Bioelectrochemistry* 171, 109283. [doi:10.1016/j.bioelechem.2026.109283](https://doi.org/10.1016/j.bioelechem.2026.109283)

Dataset. [DANDI:001517](https://dandiarchive.org/dandiset/001517/0.261006.1856), version `0.261006.1856`
(doi:10.48324/dandi.001517/0.261006.1856), "Evaluation of Di-8-ANEPPS Compatibility with Strobe Fluorescence
Microscopy for Membrane Voltage Imaging". The paper cites no version of the dandiset (its data statement says
"Data will be made available on request"), so the most recent published version is pinned. It is the only
published version, and the draft has the same 220 assets and size.

Claims. `claims/001517/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of record (main text,
Methods, legends of Figures 1 to 7; no supplement exists). The claims file decides what is tested.

Outside input. The charging time constant of Eq. (1), tau = 44.5 ns per um of cell radius, is the fit of
reference [20] (Semenov et al. 2025, Bioelectrochemistry 168, 109143, doi:10.1016/j.bioelechem.2025.109143,
PMC12553484, Fig. 5C), which the paper says it used (Methods 2.3). Reference [20] is also the source of the
background method (ROIs outside the cell near the poles), which the Methods say they follow ("Most methods were
identical to those described in detail recently [20]").

What the dandiset holds. 220 NWB files, each one set of 100 raw strobe images (40 without and 60 with a 1-us
pulse) of one CHO cell, 19 cells: on 2025-02-28 six FluoVolt cells and eight Di-8-ANEPPS cells taken through the
nine-step escalation of Fig. 2A/B and 4A/B (25 to 350 V), and on 2025-01-22 five Di-8-ANEPPS cells with a finer
escalation (25 to 300 V) that the paper does not describe. There are no 0.3-us, Gd3+, two-pulse, alternating
150/230 V/cm or Ca2+ recordings. The files hold images only: the cell outline, the field direction, the field
strength in V/cm and the pixel size are not stored and are rebuilt or assumed here (see `strobe.py`).

Choices that the outcomes rest on (each stated as an assumption in the attempt record; every reading is shown):

- Field strength. The files give the generator amplitude in volts (25, 50, 70, 100, 150, 200, 250, 300, 350 V), the
  paper V/cm (43, 150, 230, 360, 570, 780, 1000, 1200, 1400). The two are not proportional, so they are matched in
  order, step for step. A variant with the field proportional to the volts (4 V/cm per V, which keeps the paper's
  1000 to 1400 V/cm) is run for every claim that uses the field strength as a number (fig3, fig4a to fig4c, fig7b).
- Background. Not stated in the paper; reference [20] subtracts ROIs "near the poles outside of the cell" without a
  distance. Three readings are run in every claim: `far` (8 to 10 um outside the membrane, where the membrane's glow
  has decayed; adopted), `near` (2 to 4 um outside) and `none` (the claims file's reading a). After verification
  round 1 each claim also shows a sweep of 6-pixel bands from 8 to 44 pixels out and a per-pole (local) background.
- Pixel size 0.267 um (16-um camera pixels, 60x objective; the files store none, as the dump of a file's contents
  in the data check shows). A variant uses the pixel size that gives the paper's average radius of 8 um.
- Cell outline. Traced automatically; a set whose outline radius departs from its cell's median by more than 15% is
  traced again within a narrow window (four sets, three used: FluoVolt c2 at 300 V and Di-8-ANEPPS c7 at 300 and
  350 V; added after verification round 1, when the outline overlays showed the fault).
- Cells. The paper's n = 4 FluoVolt cells are read as the four with exactly the nine-set escalation (c2, c3, c4, c6;
  c5 added as a variant); its n = 5 Di-8-ANEPPS cells are not identified, so all eight are used and each Di-8-ANEPPS
  criterion is also run on the 56 five-cell subsets.
- Frames without a laser flash are left out (variant: kept).

| Claim | Status |
|---|---|
| `fig1-pole-dimming` | tested |
| `fig2a-fluovolt-saturation` | tested |
| `fig2a-fluovolt-stable` | tested |
| `fig2b-di8-climbs-to-570` | tested |
| `fig2b-di8-bleaching` | tested |
| `fig2b-di8-decline-within-set` | tested |
| `fig2cd-stable-before-electroporation` | blocked: data missing |
| `fig2c-1000-within-set` | blocked: data missing |
| `fig2d-lasting-reduction` | blocked: data missing |
| `fig3-dye-sensitivity` | tested |
| `fig3-fluovolt-nonlinear` | tested |
| `fig4a-fluovolt-limits` | tested |
| `fig4a-fluovolt-cathodic-decline` | tested |
| `fig4b-di8-limits` | tested |
| `fig4b-first-10-pulses` | tested |
| `fig4b-cathodic-asymmetry` | tested |
| `fig4c-equator-follows-cosine` | tested |
| `fig4d-short-pulses-same-limit` | blocked: data missing |
| `fig5a-gd-extends-anodic` | blocked: data missing |
| `fig5a-gd-linear` | blocked: data missing |
| `fig5a-gd-cathodic` | blocked: data missing |
| `fig5a-gd-differences` | blocked: data missing |
| `fig5b-conductivity-calibration` | tested |
| `fig5c-gd-reduces-conductivity` | blocked: data missing |
| `fig6a-ca-single-pulse` | blocked: data missing |
| `fig6b-ca-train-responders` | blocked: data missing |
| `fig6b-ca-650` | blocked: data missing |
| `fig7b-charging-during-pulse` | tested |
| `fig7ab-test-pulse-recovery` | blocked: data missing |
| `fig7c-fast-conductivity-drop` | blocked: data missing |
| `fig7c-full-reseal-mild` | blocked: data missing |

## Setup and loading

In [ ]:
import itertools
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

import strobe

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
print("dandiset", strobe.DANDISET, "version", strobe.VERSION)
assets = pd.DataFrame(strobe.list_assets()).sort_values(["subject", "start"]).reset_index(drop=True)
assets["day"] = assets.subject.str.split("-").str[1]
print(len(assets), "assets")

In [ ]:
# Stream and reduce every set of the 2025-02-28 cells (the nine-step escalations, see the data
# check); about 2 s per set. The 2025-01-22 sets are only surveyed (shapes), not analysed.
t0 = time.time()
use = assets[assets.day == "20250228"]


def _extract(row):
    s = strobe.extract_set(row.asset_id)
    s.update(row._asdict())
    return s


with ThreadPoolExecutor(4) as ex:
    S = list(ex.map(_extract, use.itertuples(index=False)))
print(f"{len(S)} sets streamed and reduced in {time.time() - t0:.0f} s")

# Re-trace (added after verification round 1; the outline overlays it asked for showed the fault): a set
# whose mean outline radius differs from the median of its cell's sets by more than 15% has had its outline
# drawn on a structure inside the cell. It is traced again with the search held within 8 pixels of the cell's median
# centre and 4 pixels of its median radius (strobe.extract_set, hint).
geo = pd.DataFrame([dict(k=k, subject=s["subject"], cy=s["cy"], cx=s["cx"], r=float(s["r"].mean())) for k, s in enumerate(S)])
med = geo.groupby("subject")[["cy", "cx", "r"]].transform("median")
redo = geo[np.abs(geo.r / med.r - 1) > 0.15]
for k in redo.k:
    row = use.iloc[k]
    s = strobe.extract_set(row.asset_id, hint=tuple(med.loc[k, ["cy", "cx", "r"]]))
    s.update(row.to_dict())
    print(f"re-traced {row.subject} {row.volts} V: radius {geo.r[k]:.1f} -> {s['r'].mean():.1f} px "
          f"(cell median {med.r[k]:.1f}); centre moved {np.hypot(s['cy'] - geo.cy[k], s['cx'] - geo.cx[k]):.1f} px")
    S[k] = s
print(f"{len(redo)} sets re-traced")

## Data check

The cells below print what was loaded and check the description against the values: counts by cell, dye and
amplitude; frame counts, image size and rate; duplicates; frames without a laser flash; cell size; the light
outside the membrane (which bears on the background readings); the field direction; the split between baseline
and pulse frames (the files have no trial table, so the split is tested by moving it); and the volts in the files
against the paper's field strengths.

In [ ]:
# ---- Data check: what is in the dandiset, and the metadata against the data ----
print("Assets by recording day, dye, pulse length and electrode distance (from the session descriptions):")
print(assets.groupby(["day", "dye", "pulse_us", "electrode_mm"]).size().to_string())
print("\nSets per cell, with the pulse amplitudes in recording order (volts, as the files give them):")
for subj, g in assets.groupby("subject", sort=False):
    print(f"  {subj:18s} {g.dye.iloc[0]:12s} n={len(g):2d}  {list(g.volts)}")

# shapes of the 2025-01-22 sets (not analysed), read from the files without their data
def _shape(aid):
    with strobe.open_asset(aid) as f:
        d = f["acquisition/StrobeImaging/data"]
        return d.shape, str(d.dtype), float(f["acquisition/StrobeImaging/starting_time"].attrs["rate"])

with ThreadPoolExecutor(8) as ex:
    jan = list(ex.map(_shape, assets[assets.day == "20250122"].asset_id))
print("\n2025-01-22 sets: shapes", sorted(set(str(j[0]) for j in jan)), "dtypes", set(j[1] for j in jan),
      "rates", set(j[2] for j in jan))
print("2025-02-28 sets: shapes", sorted(set(str((s["n_frames"],) + tuple(s["shape"])) for s in S)),
      "rates", set(s["rate"] for s in S))

# duplicates: identical image data in two files
sha = pd.Series([s["sha1"] for s in S])
print("\nsets with identical image data (2025-02-28):", int(sha.duplicated().sum()))
print("assets with identical file digests (all 220):", int(assets.asset_id.duplicated().sum()),
      "(file sizes all distinct:", assets["size"].is_unique, ")")

# frames without a laser flash
dark = np.array([s["dark"] for s in S])
print(f"\nDark frames (no laser flash): {dark.sum()} of {dark.size}; frame 0 dark in {dark[:, 0].sum()} of {len(S)} "
      f"sets; per set {dark.sum(1).min()} to {dark.sum(1).max()}; in baseline frames 1-39: {dark[:, 1:40].sum()}, "
      f"in pulse frames 40-99: {dark[:, 40:].sum()}")
fm = np.array([s["frame_mean"] for s in S])
print(f"frame mean, lit frames {np.median(fm[~dark]):.0f} (range {fm[~dark].min():.0f}-{fm[~dark].max():.0f}), "
      f"dark frames {np.median(fm[dark]):.0f} (range {fm[dark].min():.0f}-{fm[dark].max():.0f}); "
      f"camera floor (1st percentile) {np.median([s['floor'] for s in S]):.0f} counts")

# cell size, with the assumed pixel size of 16 um / 60
rad = pd.DataFrame([dict(subject=s["subject"], dye=s["dye"], a=s["radius_area"], p=s["radius_perimeter"]) for s in S])
print(f"\nCell radius (um, pixel {strobe.UM_PER_PIXEL:.3f} um): equal-area {rad.a.median():.2f} "
      f"(range {rad.a.min():.2f}-{rad.a.max():.2f}); perimeter/2pi {rad.p.median():.2f} ({rad.p.min():.2f}-{rad.p.max():.2f})")
print(rad.groupby(["dye", "subject"])[["a", "p"]].agg(["mean", "min", "max"]).round(2).to_string())
print("Paper (Methods 2.3): 'approximate average value of 8 um in CHO cells'.")

# radial profile of baseline intensity outside the membrane (for the background readings)
radial = np.array([s["radial"] for s in S])
fl = np.median([s["floor"] for s in S])
print("\nBaseline intensity at offsets from the membrane line (pixels; median over sets); camera floor", round(fl))
print(pd.Series(np.median(radial, 0).round(0), index=strobe.RADIAL_OFFSETS).to_string())
k8, k30 = list(strobe.RADIAL_OFFSETS).index(8), list(strobe.RADIAL_OFFSETS).index(30)
k0 = list(strobe.RADIAL_OFFSETS).index(0)
print(f"excess over the 40-px level at 8 px: {np.median((radial[:, k8] - radial[:, -1]) / (radial[:, k0] - radial[:, -1])):.2f} "
      f"of the membrane's; at 30 px: {np.median((radial[:, k30] - radial[:, -1]) / (radial[:, k0] - radial[:, -1])):.2f}")

In [ ]:
# What one file holds (requested in verification round 1): every group and dataset, with attributes,
# for one FluoVolt and one Di-8-ANEPPS set (the NWB schema copies under /specifications left out).
for subj in ("CHO-20250228-c2", "CHO-20250228-c7"):
    aid = assets[assets.subject == subj].asset_id.iloc[0]
    print(f"--- {subj}, asset {aid}")
    print("\n".join(strobe.nwb_tree(aid)))
with strobe.open_asset(assets.asset_id.iloc[0]) as f:
    print("\nStrobeImaging description, in full:", f["acquisition/StrobeImaging"].attrs["description"])
    print("Neurodata type of StrobeImaging:", f["acquisition/StrobeImaging"].attrs.get("neurodata_type"))

In [ ]:
# Cell outlines (requested in verification round 1): the traced membrane line (red) and the far
# background band, 30 to 36 pixels out (cyan), on the baseline image of the 25, 150 and 350 V sets of
# each cell used. Each panel is the 145 x 145-pixel crop around the cell, intensity scaled between its
# own 1st and 99.5th percentiles; the far band is drawn where it falls inside the crop, and no point of
# the membrane line lies outside its panel.
import matplotlib.pyplot as plt

OUT_CELLS = [f"CHO-20250228-c{i}" for i in (2, 3, 4, 5, 6)] + [f"CHO-20250228-c{i}" for i in range(7, 15)]
fig, axs = plt.subplots(3, len(OUT_CELLS), figsize=(2.0 * len(OUT_CELLS), 6.4))
ang = np.deg2rad(np.arange(strobe.N_POINTS))
outside = 0
for j, subj in enumerate(OUT_CELLS):
    for k, v in enumerate((25, 150, 350)):
        s = [x for x in S if x["subject"] == subj and x["volts"] == v][-1 if (subj.endswith("c5") and v == 350) else 0]
        ax = axs[k, j]
        lo, hi = np.percentile(s["crop"], [1, 99.5])
        ax.imshow(s["crop"], cmap="gray", vmin=lo, vmax=hi)
        y0, x0 = s["crop_origin"]
        for off, col in ((0, "r"), (33, "c")):
            xx = s["cx"] + (s["r"] + off) * np.cos(ang) - x0
            yy = s["cy"] - (s["r"] + off) * np.sin(ang) - y0
            ax.plot(xx, yy, col, lw=0.6)
            if off == 0:
                outside += int(((xx < 0) | (yy < 0) | (xx > 2 * strobe.CROP) | (yy > 2 * strobe.CROP)).sum())
        ax.set_xlim(0, 2 * strobe.CROP); ax.set_ylim(2 * strobe.CROP, 0)
        ax.set_xticks([]); ax.set_yticks([])
        if k == 0:
            ax.set_title(subj.split("-")[-1] + (" FV" if j < 5 else " D8"), fontsize=8)
        if j == 0:
            ax.set_ylabel(f"{v} V", fontsize=8)
plt.tight_layout()
plt.show()
print("outline points outside their panel:", outside)

# how well the line sits on the membrane: intensity on the line over the larger of the two 4 pixels inside and outside it
def ridge(s):
    rp = s["radial"]
    k0 = list(strobe.RADIAL_OFFSETS).index(0)
    return rp[k0] / max(rp[k0 - 2], rp[k0 + 2]) if k0 >= 2 else np.nan

rt = pd.DataFrame([dict(subject=s["subject"], volts=s["volts"], R=s["radius_perimeter"], ridge=ridge(s)) for s in S
                   if s["subject"] in OUT_CELLS and s["volts"] in strobe.VOLT_TO_VCM])
print("\nPerimeter radius (um, 0.267-um pixel) by cell and set (first set at each amplitude):")
print(rt.groupby(["subject", "volts"]).R.first().unstack().round(1).to_string())
print("\nMembrane line contrast (intensity on the line over the larger of 4 pixels inside or outside it; > 1 is a ridge):")
print(rt.groupby(["subject", "volts"]).ridge.first().unstack().round(2).to_string())

In [ ]:
# Field direction: not stored. Image angle (0 = right, counter-clockwise) of the pole where dF > 0,
# from the first harmonic of dF at 50-100 V (below the electroporating levels), per set.
low = [s for s in S if s["volts"] in (50, 70, 100)]
fa = pd.DataFrame([dict(subject=s["subject"], dye=s["dye"], volts=s["volts"], angle=strobe.field_angle([s])) for s in low])
print(fa.pivot_table(index="subject", columns="volts", values="angle").round(0).to_string())
AXIS = strobe.field_angle(low)
print(f"\nPooled cathode-facing direction (2025-02-28, all cells): {AXIS:.1f} deg; per-set range "
      f"{fa.angle.min():.0f}-{fa.angle.max():.0f}, median absolute deviation from it {np.median(np.abs(fa.angle - AXIS)):.1f} deg")

In [ ]:
# Clock: the files have no trial table; frames 0-39 are said to be without and 40-99 with the pulse
# (Methods 2.3). Contrast of cathodic minus anodic pole intensity after vs before a split at frame k,
# per set (sets at 50 V and above), normalised by the baseline. With the true labels the contrast should
# be largest at k = 40, and shifting the split should lower it.
def pole_diff(s):
    th = np.arange(strobe.N_POINTS)
    tr = s["trace"].astype(float)
    c = tr[:, strobe.arc(th, AXIS, 5)].mean(1)
    a = tr[:, strobe.arc(th, AXIS + 180, 5)].mean(1)
    x = (c - a) / (c + a)
    x[s["dark"]] = np.nan
    return x

ks = np.arange(20, 61, 2)
con = []
for s in S:
    if s["volts"] < 50:
        continue
    x = pole_diff(s)
    con.append([np.nanmean(x[k:]) - np.nanmean(x[1:k]) for k in ks])
con = np.array(con)
print("median contrast by split frame k (x1000):")
print(pd.Series((np.median(con, 0) * 1000).round(1), index=ks).to_string())
best = ks[np.argmax(con, axis=1)]
print(f"split giving the largest contrast: k = 40 in {np.mean(best == 40) * 100:.0f}% of {len(con)} sets; "
      f"within 38-42 in {np.mean(np.abs(best - 40) <= 2) * 100:.0f}%")

In [ ]:
# Volts in the files against the paper's field strengths. The nine volt steps are matched in order to
# the nine field strengths (an assumption, see the header). Below electroporation the polarization should
# be proportional to the field; compare a fit through the origin of pole |dF| against the paper's V/cm
# and against the volts, over 25-100 V (43-360 V/cm), for each dye and background reading.
FV_CELLS = [f"CHO-20250228-c{i}" for i in (2, 3, 4, 6)]
FV_CELLS_ALT = FV_CELLS + ["CHO-20250228-c5"]
D8_CELLS = [f"CHO-20250228-c{i}" for i in range(7, 15)]
VOLTS = list(strobe.VOLT_TO_VCM)
FIELDS = [strobe.VOLT_TO_VCM[v] for v in VOLTS]


def set_of(subject, volts):
    """The set of a cell at an amplitude, in the escalation: the first in recording order, except
    c5 (FluoVolt), whose first set is a 350 V set before its escalation, which is left out."""
    ss = [s for s in S if s["subject"] == subject and s["volts"] == volts]
    if subject == "CHO-20250228-c5" and volts == 350:
        ss = ss[1:]
    return ss[0]


for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
    for bg in strobe.BACKGROUNDS:
        y = np.array([[np.mean(np.abs(strobe.pole_values(strobe.collapsed_df(set_of(c, v), AXIS, bg))))
                       for v in VOLTS[:4]] for c in cells]).mean(0)
        out = []
        for x in (np.array(FIELDS[:4], float), np.array(VOLTS[:4], float)):
            b = (x @ y) / (x @ x)
            out.append(1 - np.sum((y - b * x) ** 2) / np.sum((y - y.mean()) ** 2))
        print(f"{name:12s} bg={bg:4s} mean pole |dF| at 25/50/70/100 V: {np.round(y, 1)}; "
              f"R2 of proportional fit on paper V/cm {out[0]:.3f}, on volts {out[1]:.3f}; "
              f"ratio 50V/25V {y[1] / y[0]:.2f} (paper fields {150 / 43:.2f}, volts 2.00)")

# Per-step ratios of the mean pole |dF| (far background) below electroporation, against the ratio of
# the paper's field strengths and of the volts (requested in verification round 1). FluoVolt approaches
# its saturation by 70 to 100 V, which flattens its later steps.
for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
    y = np.array([[np.mean(np.abs(strobe.pole_values(strobe.collapsed_df(set_of(c, v), AXIS, "far"))))
                   for v in VOLTS[:5]] for c in cells])
    for k in range(4):
        r = y[:, k + 1] / y[:, k]
        print(f"{name:12s} {VOLTS[k]:3d} -> {VOLTS[k + 1]:3d} V: ratio of means {y[:, k + 1].mean() / y[:, k].mean():.2f} "
              f"(per cell {np.round(r, 2)}); paper fields {FIELDS[k + 1] / FIELDS[k]:.2f}, volts {VOLTS[k + 1] / VOLTS[k]:.2f}")

print("\nElectrode distance in every file:", set(assets.electrode_mm), "mm; the paper (Methods 2.2): 1.6 mm.")
print("Pulse length in every file:", set(assets.pulse_us), "us; dyes:", dict(assets.dye.value_counts()))

What the checks show (numbers above):

- Counts. 220 sets of 19 cells. The 2025-02-28 cells hold the nine-step escalation: FluoVolt c2, c3, c4 and c6 exactly
  nine sets; c5 has an extra 350 V set before its escalation; c1 a finer escalation. Di-8-ANEPPS c7 to c14 nine sets
  each, c13 with a second 150 V set (the first is used). The 2025-01-22 cells (Di-8-ANEPPS) follow other steps and are
  not used. Every set has 100 frames of 512 x 512 pixels at 6 Hz, as the Methods describe (40 + 60 images). No two sets
  hold identical images.
- What a file holds. One `ImageSeries` (`StrobeImaging`) with the 100 frames, its rate and a general description,
  plus subject, session and lab fields. There is no `ImagingPlane`, `Device`, field of view, pixel size or grid
  spacing, no ROI or segmentation, and nothing under `processing` or `stimulus`. The pixel size, the outline, the
  field direction and the field strength in V/cm therefore have to be rebuilt or assumed.
- The dandiset description speaks of Di-8-ANEPPS only ("replacing FluoVolt"), but 68 sets are FluoVolt.
- Every session description gives an electrode distance of 1.00 mm; the paper and reference [20] give 1.6 mm.
- Frames without a laser flash: frame 0 of almost every set (137 of 141), and a few others, have a mean near the
  camera floor. The files do not flag them. Apart from frame 0 they are five times as frequent among the pulse frames
  (257 of 8460) as among the baseline frames (33 of 5499).
- Cell outline. The first tracing put the outline of four sets on another structure: Di-8-ANEPPS c7 at 300 and 350 V
  on a bright body inside the cell (radius 22 against 41 pixels), FluoVolt c2 at 300 V on a neighbouring cell (centre
  78 pixels away) and c1 at 400 V (not used). These four are traced again (loading cell), after which every used set's
  radius is within 15% of its cell's median, the outline lies on a ridge of intensity in every set (contrast 1.04 to
  1.22 over the intensity 4 pixels inside or outside), and the overlays show it on the membrane. The least sharp ridges
  are Di-8-ANEPPS c13 (1.04 to 1.08). In the overlays the FluoVolt outlines follow the membrane closely; several
  Di-8-ANEPPS outlines (c7, c10, c12, c13, c14) are wavy and cut inside the membrane along part of the perimeter, where
  bright bodies inside the cell lie close to it. Those stretches can lower or raise the trace intensity there and add
  noise; they are a difference from the paper's hand-drawn trace. The adopted outline is kept; a variant traced on the
  outer membrane (after verification round 2, before the Summary) shows the Di-8-ANEPPS outcomes do not change.
- Cell radius with a 0.267-um pixel is 7.8 to 15.1 um by cell (mean over sets), larger than the paper's "approximate
  average value of 8 um"; consistent with a smaller pixel or larger cells, which the data cannot tell apart.
- Light outside the membrane: 8 pixels (2 um) out, the intensity above the 40-pixel level is still about a third of
  the membrane's; 16 to 18 pixels out about a sixth; 30 pixels out a few percent. A background taken near the membrane
  removes part of the membrane's own out-of-focus light; this is why the `far` reading is adopted.
- Field direction: the depolarized pole lies at the same image angle in every cell and set (the per-set angles
  above), so one axis is used for all sets.
- Clock: the polarization contrast is largest when the split is placed at frame 40 in nearly every set, and falls
  when the split is moved, so the 40/60 split of the Methods matches the data.
- Volts against V/cm. Over the four lowest steps the proportional fits do not separate the two scales (R2 on volts
  as high or higher, distorted for FluoVolt by its saturation). The first step does: from 25 to 50 V the mean pole
  |dF| triples (FluoVolt 3.05, Di-8-ANEPPS 2.90, far background), against 3.49 for the paper's 43 to 150 V/cm and 2.00
  for the volts. The later Di-8-ANEPPS steps (1.49, 1.35 and 1.23) are at or below the ratios of both scales (paper
  1.53, 1.57 and 1.58; volts 1.40, 1.43 and 1.50), and FluoVolt is near saturation by then. The data are consistent with the in-order matching and
  argue against a field proportional to the volts at the low end; the matching stays an assumption.

## Shared tables

Collapsed pole dF for every cell, field strength and background reading, and helpers used by the claims.

In [ ]:
# ---- Tables shared by the claims ----
# Collapsed pole dF (Methods 2.3, Fig. 3 onward) per cell, field strength and background reading;
# also with the first 10 pulse images only, and with the dark frames kept (variant).
# Background readings: the three adopted ones (far, near, none), the 6-pixel bands from 8 to 44 pixels
# outside the membrane, and the per-pole (local) near and far backgrounds (see strobe.py).
ALL_BG = list(strobe.BACKGROUNDS) + list(strobe.SWEEP) + ["near_local", "far_local"]
rows = []
for subj in FV_CELLS_ALT + D8_CELLS:
    for v in VOLTS:
        s = set_of(subj, v)
        for bg in ALL_BG:
            c, a = strobe.pole_values(strobe.collapsed_df(s, AXIS, bg))
            c10, a10 = strobe.pole_values(strobe.collapsed_df(s, AXIS, bg, pulse_frames=slice(0, 10)))
            ck, ak = strobe.pole_values(strobe.collapsed_df(s, AXIS, bg, keep_dark=True))
            rows.append(dict(subject=subj, dye=s["dye"], volts=v, E=strobe.VOLT_TO_VCM[v], bg=bg, cath=c, anod=a,
                             cath10=c10, anod10=a10, cath_dark=ck, anod_dark=ak,
                             R_area=s["radius_area"], R_perim=s["radius_perimeter"]))
P = pd.DataFrame(rows)


def means(cells, bg, cols=("cath", "anod")):
    """Mean across cells per field strength (and the s.e.)."""
    d = P[(P.subject.isin(cells)) & (P.bg == bg)]
    g = d.groupby("E")[list(cols)]
    return g.mean(), g.sem()


def per_image(subj, v, bg, keep_dark=False):
    return strobe.per_image_pole_df(set_of(subj, v), AXIS, bg, keep_dark)


def show_means(cells, label):
    for bg in strobe.BACKGROUNDS:
        m, se = means(cells, bg)
        print(f"{label}, collapsed pole dF (%), background {bg}: mean (s.e.) over {len(cells)} cells")
        print(pd.DataFrame({"cath": m.cath.round(1), "cath_se": se.cath.round(1), "anod": m.anod.round(1),
                            "anod_se": se.anod.round(1)}).T.to_string())


D8_SUBSETS = list(itertools.combinations(D8_CELLS, 5))


def subset_share(test):
    """Share of the 56 five-cell subsets of the 8 Di-8-ANEPPS cells under which test(cells) holds
    (the paper's n = 5 cells are not identified)."""
    r = [bool(test(list(c))) for c in D8_SUBSETS]
    return f"{sum(r)}/{len(r)}"


def sweep(fn):
    """fn(bg) -> dict of results, under every background reading of ALL_BG (one row each)."""
    return pd.DataFrame({bg: fn(bg) for bg in ALL_BG}).T


def ttest(x, y, paired):
    x, y = np.asarray(x, float), np.asarray(y, float)
    return (stats.ttest_rel(x, y) if paired else stats.ttest_ind(x, y)).pvalue

## fig1-pole-dimming

> With Di-8-ANEPPS and 1-us pulses, polarization grows with field strength up to 570 V/cm; at 780 V/cm the cathodic pole stops increasing and dims, and at 1000 to 1400 V/cm the poles show reduced or absent polarization while regions nearer the equator stay polarized and may polarize more than the poles.

Criterion: the eight Di-8-ANEPPS escalation cells (the Fig. 1 cell is not identified), collapsed profiles.
(1) mean cathodic pole dF at 780 not above 570 V/cm; (2) at 1400 V/cm the largest mean dF of the cathode-facing
hemisphere outside the +-5 degree polar arc. All background readings are run.

In [ ]:
# fig1-pole-dimming: Di-8-ANEPPS 1-us escalation cells, collapsed profiles. (1) mean cathodic pole dF
# at 780 not above 570 V/cm; (2) at 1400 V/cm the largest mean dF of the cathode-facing hemisphere
# lies outside the +-5 degree polar arc.
th = np.arange(strobe.N_POINTS)
hemi = strobe.arc(th, 0, 90)


def fig1_test(cells, bg):
    m, _ = means(cells, bg)
    c1 = m.cath[780] <= m.cath[570]
    prof = np.mean([strobe.collapsed_df(set_of(c, 350), AXIS, bg) for c in cells], 0)
    hp = np.where(hemi, prof, -np.inf)
    k = int(np.argmax(hp))
    ang = (k + 180) % 360 - 180
    return c1, abs(ang) > strobe.POLE_HALF_WIDTH, m.cath[570], m.cath[780], ang, prof[k], prof[strobe.arc(th, 0, 5)].mean()


for bg in strobe.BACKGROUNDS:
    c1, c2, a570, a780, ang, pk, pole = fig1_test(D8_CELLS, bg)
    print(f"bg={bg:4s}: cathodic 570 {a570:.1f}%, 780 {a780:.1f}% -> (1) {c1}; at 1400 V/cm the cathode-facing "
          f"maximum {pk:.1f}% is at {ang:+d} deg (pole arc mean {pole:.1f}%) -> (2) {c2}; both: {c1 and c2}; "
          f"5-cell subsets passing {subset_share(lambda c: all(fig1_test(c, bg)[:2]))}")
# per-cell profile maxima at 1400 V/cm, far background
for c in D8_CELLS:
    prof = strobe.collapsed_df(set_of(c, 350), AXIS, "far")
    k = int(np.argmax(np.where(hemi, prof, -np.inf)))
    print(f"  {c}: 1400 V/cm cathode-facing max {prof[k]:.1f}% at {(k + 180) % 360 - 180:+d} deg, pole {prof[strobe.arc(th, 0, 5)].mean():.1f}%")

In [ ]:
# Variant (verification round 1): every background reading, including the distance sweep and the
# per-pole (local) backgrounds.
print(sweep(lambda bg: dict(zip(("c1_780_not_above_570", "c2_max_off_pole", "cath570", "cath780", "max_angle", "max_dF", "pole_dF"),
                                fig1_test(D8_CELLS, bg)))).to_string())

**Result: reproduced.** Under every background reading the cathodic pole is lower at 780 than at 570 V/cm (far 38.6
against 41.7%), and at 1400 V/cm the largest mean dF of the cathode-facing hemisphere lies 65 degrees from the pole
(far 21.7% at -65 degrees, against 7.5% at the pole; near and none also at -65). Both conditions hold at every
distance of the sweep and with the local backgrounds. Under the far background all 56 five-cell subsets pass (45 of
56 near, 51 of 56 none). In each of the 8 cells the 1400 V/cm maximum lies 43 to 76 degrees from the pole.

## fig2a-fluovolt-saturation

> With FluoVolt, pole dF rose with field strength until it saturated at 66% (255 mV) at 360 V/cm at the cathodic pole and -83% (-320 mV) at 570 V/cm at the anodic pole; stronger fields did not increase the anodic response and decreased the cathodic one.

Criterion: per-image pole dF (the Fig. 2 measure) over the 60 pulse images, per cell then across the four FluoVolt
cells. Both values within tolerance (cathodic 66 +- 6.6% at 360 V/cm, anodic -83 +- 8.3% at 570 V/cm), and the
three shape conditions.

In [ ]:
# fig2a-fluovolt-saturation: per-image pole dF (Fig. 2 measure) over the 60 pulse images, per cell then
# across the 4 FluoVolt cells.
def fig2_table(cells, bg, keep_dark=False):
    out = {}
    for c in cells:
        for v in VOLTS:
            ca, an = per_image(c, v, bg, keep_dark)
            out[(c, strobe.VOLT_TO_VCM[v])] = (np.nanmean(ca[40:]), np.nanmean(an[40:]))
    d = pd.DataFrame(out, index=["cath", "anod"]).T
    d.index.names = ["subject", "E"]
    return d


def fig2a_test(cells, bg, keep_dark=False):
    d = fig2_table(cells, bg, keep_dark).groupby("E").mean()
    c, a = d.cath, d.anod
    conds = dict(cath360_in_tol=abs(c[360] - 66) <= 6.6, anod570_in_tol=abs(a[570] - (-83)) <= 8.3,
                 cath570_780_not_above=bool(c[570] <= c[360] + 6.6 and c[780] <= c[360] + 6.6),
                 anod780_1400_not_more_neg=bool(all(a[E] >= a[570] - 8.3 for E in (780, 1000, 1200, 1400))),
                 cath1000_1400_below_360=bool(all(c[E] < c[360] for E in (1000, 1200, 1400))))
    return d, conds


for cells, lab in ((FV_CELLS, "4 cells c2,c3,c4,c6"), (FV_CELLS_ALT, "5 cells (+c5)")):
    for bg in strobe.BACKGROUNDS:
        d, conds = fig2a_test(cells, bg)
        print(f"{lab}, bg={bg}: cath {d.cath.round(1).to_dict()}")
        print(f"{'':>24} anod {d.anod.round(1).to_dict()}")
        print(f"{'':>24} {conds} -> all: {all(conds.values())}")
d, conds = fig2a_test(FV_CELLS, "far", keep_dark=True)
print("variant, dark frames kept (4 cells, far):", round(d.cath[360], 1), round(d.anod[570], 1), all(conds.values()))

In [ ]:
# Variant (verification round 1): the background distance. The two values and the shape conditions under
# every background reading, including 6-pixel bands from 8 to 44 pixels out and the per-pole backgrounds.
def _f2a(bg):
    d, conds = fig2a_test(FV_CELLS, bg)
    return dict(cath360=round(d.cath[360], 1), anod570=round(d.anod[570], 1), cath1400=round(d.cath[1400], 1), **conds,
                all=all(conds.values()))
print(sweep(_f2a).to_string())

**Result: not reproduced** (changed from partially reproduced after verification round 1). The two values the
criterion requires miss under the adopted far background: cathodic at 360 V/cm 49.4% against 66 +- 6.6, anodic at
570 V/cm -69.8% against -83 +- 8.3. The shape conditions hold under it: the cathodic mean at 570 and 780 V/cm does not
exceed that at 360 V/cm (48.1 and 37.4%), the anodic pole grows beyond 570 V/cm by no more than the tolerance (-72.2% at 780 V/cm,
2.4 points beyond -69.8%, then to -61.9% at 1400 V/cm), and the
cathodic pole is lower at 1000 to 1400 V/cm (36.6 to 40.9%). The values scale with the background distance: near
78.5 and -88.6% (the anodic within tolerance, but 1400 V/cm then rises to 80.4%), none 23.7 and -37.6%. In the
sweep one band, 14 to 20 pixels (4 to 5 um) out, meets every condition (61.3 and -78.1%), where the membrane's glow is
still about a sixth of its excess; no band from 20 pixels out does. With c5 added, 55.2 and -71.2% (far).

## fig2a-fluovolt-stable

> FluoVolt measurements were stable without pulses (no significant bleaching or photosensitization), and the response within each set stayed stable through the 60 pulses, even at 1000 V/cm and above.

Claim of absence, judged on the 2-tailed paired t-test of the Fig. 2 legend (n = 4 cells, p < 0.05). Start against
end of the 60 pulse images (sets at 1000 to 1400 V/cm) and of the 40 baseline images (all nine sets), readings
(a) first and last image, (b) first 10 and last 10.

In [ ]:
# fig2a-fluovolt-stable (claim of absence): per set and pole, start against end of the 60 pulse images
# and of the 40 baseline images, 2-tailed paired t-test over the 4 cells at p < 0.05.
# Readings: (a) first and last image, (b) first 10 and last 10 images (lit frames only).
def start_end(x, reading):
    idx = np.where(~np.isnan(x))[0]
    if reading == "a":
        return x[idx[0]], x[idx[-1]]
    return np.nanmean(x[idx[:10]]), np.nanmean(x[idx[-10:]])


def within_set(cells, bg, reading, part):
    """Per cell, E and pole: (start, end) of the pulse ('pulse') or baseline ('base') images."""
    out = []
    for c in cells:
        for v in VOLTS:
            ca, an = per_image(c, v, bg)
            for pole, x in (("cath", ca), ("anod", an)):
                seg = x[40:] if part == "pulse" else x[:40]
                st, en = start_end(seg, reading)
                out.append(dict(subject=c, E=strobe.VOLT_TO_VCM[v], pole=pole, start=st, end=en))
    return pd.DataFrame(out)


def change_tests(d):
    r = []
    for (E, pole), g in d.groupby(["E", "pole"]):
        r.append(dict(E=E, pole=pole, start=g.start.mean(), end=g.end.mean(), change=(g.end - g.start).mean(),
                      p=stats.ttest_rel(g.start, g.end).pvalue))
    return pd.DataFrame(r)


res = {}
for bg in strobe.BACKGROUNDS:
    for reading in ("a", "b"):
        tp = change_tests(within_set(FV_CELLS, bg, reading, "pulse"))
        tb = change_tests(within_set(FV_CELLS, bg, reading, "base"))
        sig_p = tp[(tp.E >= 1000) & (tp.p < 0.05)]
        sig_b = tb[tb.p < 0.05]
        res[(bg, reading)] = len(sig_p) == 0 and len(sig_b) == 0
        print(f"bg={bg:4s} reading ({reading}): pulse sets 1000-1400 with p<0.05: {len(sig_p)}; baselines with p<0.05: "
              f"{len(sig_b)} of {len(tb)} -> no change: {res[(bg, reading)]}")
        if bg == "far":
            print("  pulse images, change (end - start, % dF) and p:")
            print(tp.pivot(index="pole", columns="E", values="change").round(1).to_string())
            print(tp.pivot(index="pole", columns="E", values="p").round(3).to_string())
            print("  baseline images, change and p:")
            print(tb.pivot(index="pole", columns="E", values="change").round(1).to_string())
            print(tb.pivot(index="pole", columns="E", values="p").round(3).to_string())
        else:
            print("  significant:", sig_p[["E", "pole", "change", "p"]].round(3).values.tolist(),
                  sig_b[["E", "pole", "change", "p"]].round(3).values.tolist())

In [ ]:
# Variant (verification round 1): every background reading; number of tests (of 6 pulse and 18 baseline)
# with p < 0.05 under each start/end reading. With 24 tests at p < 0.05, about 1.2 are expected by
# chance when nothing changes.
def _stab(bg):
    out = {}
    for reading in ("a", "b"):
        tp = change_tests(within_set(FV_CELLS, bg, reading, "pulse"))
        tb = change_tests(within_set(FV_CELLS, bg, reading, "base"))
        sig = pd.concat([tp[(tp.E >= 1000) & (tp.p < 0.05)], tb[tb.p < 0.05]])
        out[f"n_sig_{reading}"] = len(sig)
        out[f"largest_change_{reading}"] = round(float(sig.change.abs().max()), 1) if len(sig) else 0.0
    return out
print(sweep(_stab).to_string())

**Result: not reproduced.** Under the adopted far background both readings find changes: reading (a) one pulse set
(1400 V/cm anodic, -3.2%, p = 0.043) and one baseline (1000 V/cm anodic, -7.0%, p = 0.005); reading (b) two pulse
sets (1200 V/cm cathodic +5.5%, p = 0.049; 1400 V/cm anodic +2.5%, p = 0.037) and three baselines (150 V/cm cathodic
-3.9%, 360 V/cm anodic +3.7% and cathodic -4.2%, p 0.000 to 0.008). Without background both readings also find
changes; with the near background reading (a) finds none and (b) three. In the sweep reading (b) finds 3 to 6
changes at every distance, reading (a) none only at 8 to 20 pixels. Each reading makes 24 tests at p < 0.05, so about
one is expected by chance and at least one in about 70% of stable recordings; the changes found are a few percent dF
(largest 7.0%), small against the 40 to 70% responses. Whether the criterion should correct for the number of tests
is asked of a reader (questions/001517.yaml).

## fig2b-di8-climbs-to-570

> Di-8-ANEPPS dF kept increasing with field strength up to 570 V/cm at both poles.

Criterion: per-image pole dF over the 60 pulse images, per cell then across the Di-8-ANEPPS cells; mean |dF| at
each pole increases at every step from 43 to 570 V/cm.

In [ ]:
# fig2b-di8-climbs-to-570: per-image pole dF averaged over the 60 pulse images, per cell then across the
# Di-8-ANEPPS cells; mean |dF| at each pole increases at every step from 43 to 570 V/cm.
def climbs(cells, bg):
    d = fig2_table(cells, bg).groupby("E").mean().abs()
    steps = [43, 150, 230, 360, 570]
    return d, all(np.diff(d.cath[steps]) > 0) and all(np.diff(d.anod[steps]) > 0)


T2B = {bg: fig2_table(D8_CELLS, bg) for bg in strobe.BACKGROUNDS}
for bg in strobe.BACKGROUNDS:
    d, ok = climbs(D8_CELLS, bg)
    print(f"bg={bg:4s} mean |dF| cath {d.cath.round(1).to_dict()}\n{'':10s} anod {d.anod.round(1).to_dict()}")
    print(f"{'':10s} increases at every step 43-570 at both poles: {ok}; 5-cell subsets passing "
          f"{subset_share(lambda c: climbs(c, bg)[1])}")
print("per cell, far background, cathodic (rows) by field:")
print(T2B["far"].cath.unstack().round(1).to_string())
print("anodic:")
print(T2B["far"].anod.unstack().round(1).to_string())

In [ ]:
# Variant (verification round 1): every background reading.
print(sweep(lambda bg: dict(climbs_both_poles=climbs(D8_CELLS, bg)[1])).T.to_string())

**Result: reproduced.** Under every background reading, including every band of the sweep and the local
backgrounds, both poles increase at every step from 43 to 570 V/cm (far: cathodic 1.1, 15.9, 23.3, 33.3, 41.5%;
anodic 9.1, 17.2, 25.8, 32.6, 39.4%), and all 56 five-cell subsets pass under each of far, near and none.

## fig2b-di8-bleaching

> Di-8-ANEPPS responses were somewhat affected by bleaching, most visibly at 43 V/cm, where bleaching added about 5% to dF at the anodic pole and took about 5% from it at the cathodic pole.

Criterion: 43 V/cm set, per-image pole dF, mean across the Di-8-ANEPPS cells; reading (a) per pole, last 10 minus
first 10 pulse images; reading (b) the mean of the two poles' set-mean dF. Reproduced when one reading gives -5 +-
0.5% (at each pole for reading a).

In [ ]:
# fig2b-di8-bleaching: 43 V/cm set, per-image pole dF, mean across the Di-8-ANEPPS cells. Reading (a): per pole,
# mean dF of the last 10 minus the first 10 pulse images; reading (b): the mean of the cathodic and anodic
# set-mean dF over the 60 pulse images. Expected -5 +- 0.5 (% dF).
def bleach(cells, bg, keep_dark=False):
    a_c, a_a, b = [], [], []
    for c in cells:
        ca, an = per_image(c, 25, bg, keep_dark)
        for x, store in ((ca, a_c), (an, a_a)):
            p = x[40:]
            store.append(np.nanmean(p[-10:]) - np.nanmean(p[:10]))
        b.append((np.nanmean(ca[40:]) + np.nanmean(an[40:])) / 2)
    return np.mean(a_c), np.mean(a_a), np.mean(b), (np.array(a_c), np.array(a_a), np.array(b))


for bg in strobe.BACKGROUNDS:
    ac, aa, b, raw = bleach(D8_CELLS, bg)
    ok_a = abs(ac + 5) <= 0.5 and abs(aa + 5) <= 0.5
    ok_b = abs(b + 5) <= 0.5
    print(f"bg={bg:4s} (a) last10-first10 cath {ac:+.2f}, anod {aa:+.2f} -> {ok_a}; (b) common offset {b:+.2f} -> {ok_b}")
    if bg == "far":
        print("   per cell (a) cath", raw[0].round(1), "anod", raw[1].round(1), "(b)", raw[2].round(1))
ac, aa, b, _ = bleach(D8_CELLS, "far", keep_dark=True)
print(f"variant, dark frames kept (far): (a) {ac:+.2f} {aa:+.2f}; (b) {b:+.2f}")
# the drift through the whole 43 V/cm set, for reference: last 10 pulse images against the baseline
for bg in strobe.BACKGROUNDS:
    v = [np.nanmean(np.r_[per_image(c, 25, bg)[0][-10:], per_image(c, 25, bg)[1][-10:]]) for c in D8_CELLS]
    print(f"bg={bg:4s} mean of both poles over the last 10 pulse images: {np.mean(v):+.2f}% (per cell {np.round(v, 1)})")

In [ ]:
# Variants (verification round 1): every background reading, and the share of the 56 five-cell subsets
# passing under each reading (the paper's five cells are not identified).
def _bl(bg):
    ac, aa, b, _ = bleach(D8_CELLS, bg)
    return dict(a_cath=round(ac, 2), a_anod=round(aa, 2), b_common=round(b, 2),
                a_ok=abs(ac + 5) <= 0.5 and abs(aa + 5) <= 0.5, b_ok=abs(b + 5) <= 0.5)
print(sweep(_bl).to_string())
for bg in ("far", "near", "none"):
    print(f"bg={bg:4s}: 5-cell subsets passing (b) {subset_share(lambda c: abs(bleach(c, bg)[2] + 5) <= 0.5)}, "
          f"(a) {subset_share(lambda c: abs(bleach(c, bg)[0] + 5) <= 0.5 and abs(bleach(c, bg)[1] + 5) <= 0.5)}, "
          f"either {subset_share(lambda c: abs(bleach(c, bg)[2] + 5) <= 0.5 or (abs(bleach(c, bg)[0] + 5) <= 0.5 and abs(bleach(c, bg)[1] + 5) <= 0.5))}")

**Result: reproduced, under reading (b), and fragile.** The offset common to both poles is -5.11% with the far
background (-5.00% near, -2.63% without background); it is within tolerance at every band of the sweep (-4.58 to
-5.12%) and with the far local background (-5.32%), not with the near local one (-6.30%). Reading (a) misses at the
anodic pole under every background (far: cathodic -5.32, anodic -3.39%). The result is fragile: 38 of 56 five-cell
subsets pass under the far background (13 near, 0 none), and with the frames without a laser flash kept reading (b)
gives -2.18%. Over the last 10 pulse images both poles sit 7.2% below the baseline (far), so dF drifts downward through
the 43 V/cm set at both poles, as the text describes.

## fig2b-di8-decline-within-set

> With Di-8-ANEPPS, responses to pulses of 780 V/cm and above decreased gradually within each set of 60 images.

Criterion: cathodic |dF| at the start and end of the 60 pulse images, mean across the Di-8-ANEPPS cells; readings
(a) first and last image, (b) first 10 and last 10; every set from 780 to 1400 V/cm must end below its start under
both readings. The anodic pole is a variant.

In [ ]:
# fig2b-di8-decline-within-set: cathodic |dF| at the start and end of the 60 pulse images, mean across cells,
# readings (a) first/last image, (b) first 10/last 10; every set from 780 to 1400 must end below its start.
for bg in strobe.BACKGROUNDS:
    out = {}
    for reading in ("a", "b"):
        d = within_set(D8_CELLS, bg, reading, "pulse")
        d["start"], d["end"] = d.start.abs(), d.end.abs()
        t = change_tests(d)
        out[reading] = t
        cat = t[(t.pole == "cath") & (t.E >= 780)]
        an = t[(t.pole == "anod") & (t.E >= 780)]
        print(f"bg={bg:4s} ({reading}) cathodic |dF| start->end: "
              + ", ".join(f"{int(r.E)}: {r.start:.1f}->{r.end:.1f} (p={r.p:.3f})" for r in cat.itertuples())
              + f" -> all decline: {bool((cat.end < cat.start).all())}")
        print(f"{'':16s} anodic variant: " + ", ".join(f"{int(r.E)}: {r.start:.1f}->{r.end:.1f} (p={r.p:.3f})" for r in an.itertuples())
              + f" -> all decline: {bool((an.end < an.start).all())}")
    if bg == "far":
        for reading in ("a", "b"):
            def _t(c, reading=reading):
                d = within_set(c, "far", reading, "pulse")
                d = d[(d.pole == "cath") & (d.E >= 780)]
                g = d.groupby("E")[["start", "end"]].mean().abs()
                return bool((g.end < g.start).all())
            print(f"  far ({reading}): 5-cell subsets passing {subset_share(_t)}")

In [ ]:
# Bleaching control (verification round 1). Bleaching lowers dF at both poles through a set, which by
# itself makes cathodic dF fall and anodic |dF| rise. Start and end of the pulse images at every field
# strength (far background), with the common-mode change (mean of the cathodic and anodic signed changes)
# and the cathodic change net of it.
for reading in ("a", "b"):
    d = within_set(D8_CELLS, "far", reading, "pulse")
    d["chg"] = d.end - d.start
    w = d.pivot_table(index="E", columns="pole", values=["start", "end", "chg"])
    t = pd.DataFrame({"cath_start": w["start"]["cath"], "cath_end": w["end"]["cath"], "anod_start": w["start"]["anod"],
                      "anod_end": w["end"]["anod"], "cath_change": w["chg"]["cath"],
                      "common_change": (w["chg"]["cath"] + w["chg"]["anod"]) / 2})
    t["cath_net_of_common"] = t.cath_change - t.common_change
    print(f"reading ({reading}), far background, signed dF (%):")
    print(t.round(1).T.to_string())
# every background reading, criterion as written
def _dec(bg):
    out = {}
    for reading in ("a", "b"):
        d = within_set(D8_CELLS, bg, reading, "pulse")
        d = d[(d.pole == "cath") & (d.E >= 780)]
        g = d.groupby("E")[["start", "end"]].mean().abs()
        out[reading] = bool((g.end < g.start).all())
    return out
print(sweep(_dec).T.to_string())

**Result: reproduced.** Under the adopted far background the cathodic response falls within every set from 780 to
1400 V/cm under both readings (reading b: 41.1 to 34.3, 37.0 to 25.2, 25.5 to 15.3, 15.6 to 8.7%; paired p 0.017,
0.001, 0.002, 0.092). It holds under both readings at every distance of the sweep, without background and with the
near background; with the near local background reading (a) fails. 44 (a) and 52 (b) of 56 five-cell subsets pass
(far). The anodic pole does not decline in every set (780 V/cm rises). Bleaching control: in the sets below
electroporation the cathodic dF also falls through the set (reading b: -3.5 to -4.6% at 230 to 570 V/cm, with a
common-mode change of both poles of -1.7 to -3.4%), so the fall at 780 V/cm (-6.8%, -3.3% net of the common mode
against -2.6% at 570) is about the size of the drift; the falls at 1000 to 1400 V/cm (-9.5 to -11.8%, net -6.0 to
-7.5%) exceed it.

## fig3-dye-sensitivity

> Fitted linearly through the origin over -150 to +150 mV IMPC, FluoVolt changes by 26% and Di-8-ANEPPS by 9.1% dF per 100 mV, with correlation coefficients 0.96 and 0.78.

Criterion: every perimeter point of every cell in the 43 to 360 V/cm sets, dF against IMPC (Eq. 1, tau from
reference [20]), fit through the origin over -150 to +150 mV. Slopes within 10% of 26 and 9.1% per 100 mV, and the
correlation coefficients (Pearson, or the root of the uncentred R2) within 10% of 0.96 and 0.78.

In [ ]:
# fig3-dye-sensitivity: every perimeter point of every cell in the 43-360 V/cm sets, dF against IMPC
# (Eq. 1, tau = 44.5 ns/um x R from reference [20]); points with |IMPC| <= 150 mV fit through the origin.
# Correlation readings: (a) Pearson r in that range, (b) sqrt of the uncentred R2 of the fit through the origin.
# Radius readings: (a) perimeter / 2 pi, (b) equal-area radius.
def calib_points(cells, bg, rkey, vmap=strobe.VOLT_TO_VCM, scale=1.0):
    xs, ys = [], []
    for c in cells:
        for v in VOLTS[:4]:
            s = set_of(c, v)
            xs.append(strobe.impc_profile(vmap[v], s[rkey] * scale))
            ys.append(strobe.collapsed_df(s, AXIS, bg))
    return np.concatenate(xs), np.concatenate(ys)


def fit0(x, y, lim=150):
    m = np.abs(x) <= lim
    x, y = x[m], y[m]
    b = (x @ y) / (x @ x)
    return b * 100, np.corrcoef(x, y)[0, 1], (x @ y) / np.sqrt((x @ x) * (y @ y)), m.sum()


EXP3 = {"FluoVolt": (26, 0.96), "Di-8-ANEPPS": (9.1, 0.78)}
CAL = {}
rows = []
for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
    for bg in strobe.BACKGROUNDS:
        for rkey in ("radius_perimeter", "radius_area"):
            x, y = calib_points(cells, bg, rkey)
            slope, r_p, r_u, n = fit0(x, y)
            CAL[(name, bg, rkey)] = (x, y, slope)
            es, er = EXP3[name]
            rows.append(dict(dye=name, bg=bg, radius=rkey.split("_")[1], n=n, slope=round(slope, 2),
                             pearson=round(r_p, 3), uncentred=round(r_u, 3),
                             slope_ok=abs(slope - es) <= 0.1 * es, r_ok_a=abs(r_p - er) <= 0.1 * er,
                             r_ok_b=abs(r_u - er) <= 0.1 * er, IMPC_range=f"{x.min():.0f} to {x.max():.0f}"))
T3 = pd.DataFrame(rows)
print(T3.to_string())
for bg in strobe.BACKGROUNDS:
    t = T3[T3.bg == bg]
    ok = all(t.groupby("radius").apply(lambda g: g.slope_ok.all() and (g.r_ok_a.all() or g.r_ok_b.all())))
    print(f"bg={bg}: both slopes and correlations within tolerance (each radius reading): {ok}")
print("Di-8-ANEPPS slope over the 5-cell subsets (far, perimeter radius): ",
      np.round(np.percentile([fit0(*calib_points(list(c), 'far', 'radius_perimeter'))[0] for c in D8_SUBSETS], [0, 50, 100]), 2))

In [ ]:
# Variant: a pixel size giving the paper's average radius of 8 um (Methods 2.3), which scales every IMPC by
# 8 / (mean radius here). Not the adopted reading (the pixel size is taken from the camera and objective).
mean_r = np.mean([set_of(c, v)["radius_perimeter"] for c in FV_CELLS + D8_CELLS for v in VOLTS[:4]])
scale = 8.0 / mean_r
print(f"\nmean radius (perimeter reading) {mean_r:.2f} um; pixel size giving 8 um: {strobe.UM_PER_PIXEL * scale:.3f} um")
for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
    for bg in strobe.BACKGROUNDS:
        xs, ys = [], []
        for c in cells:
            for v in VOLTS[:4]:
                s = set_of(c, v)
                xs.append(strobe.impc_profile(strobe.VOLT_TO_VCM[v], s["radius_perimeter"] * scale))
                ys.append(strobe.collapsed_df(s, AXIS, bg))
        sl, rp, ru, n = fit0(np.concatenate(xs), np.concatenate(ys))
        print(f"  8-um variant {name:12s} bg={bg:4s}: slope {sl:.2f} %/100 mV (expected {EXP3[name][0]}), Pearson {rp:.3f}")

In [ ]:
# Variants (verification round 1). (1) Field proportional to the volts (4 V/cm per V, strobe.VOLT_TO_VCM_PROP),
# adopted and 8-um pixel. (2) Every background reading with both pixel sizes (perimeter radius).
for vlab, vmap in (("in order (adopted)", strobe.VOLT_TO_VCM), ("proportional", strobe.VOLT_TO_VCM_PROP)):
    for plab, sc in (("0.267 um", 1.0), ("8-um", scale)):
        r = []
        for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
            sl, rp, ru, n = fit0(*calib_points(cells, "far", "radius_perimeter", vmap, sc))
            r.append(f"{name} slope {sl:.2f}, Pearson {rp:.3f}")
        print(f"volts {vlab:18s} pixel {plab:8s} far: " + "; ".join(r))
s_fv, s_d8 = (fit0(*calib_points(c, "far", "radius_perimeter"))[0] for c in (FV_CELLS, D8_CELLS))
print(f"ratio of the slopes FluoVolt / Di-8-ANEPPS (far, adopted): {s_fv / s_d8:.2f}; paper 26 / 9.1 = {26 / 9.1:.2f}")


def _f3(bg):
    out = {}
    for plab, sc in (("px267", 1.0), ("px8um", scale)):
        ok = True
        for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
            sl, rp, ru, n = fit0(*calib_points(cells, bg, "radius_perimeter", strobe.VOLT_TO_VCM, sc))
            out[f"{plab}_{name[:2]}_slope"] = round(sl, 2)
            out[f"{plab}_{name[:2]}_r"] = round(rp, 3)
            es, er = EXP3[name]
            ok &= abs(sl - es) <= 0.1 * es and (abs(rp - er) <= 0.1 * er or abs(ru - er) <= 0.1 * er)
        out[f"{plab}_all_in_tol"] = bool(ok)
    return out
print(sweep(_f3).to_string())

**Result: partially reproduced** (changed from not reproduced after verification round 1). With the adopted far
background and 0.267-um pixel the Pearson correlations match (0.964 and 0.762, against 0.96 and 0.78) and the ratio of
the two slopes is close to the paper's (2.92 against 2.86), but the slopes are 21.6 (FluoVolt) and 7.4% per 100 mV
(Di-8-ANEPPS), 17 and 19% below the paper's 26 and 9.1. The radius readings do not change this (equal-area 21.6 and
7.5). The slopes scale with the assumed pixel size and the background: with the pixel that gives the paper's average
radius of 8 um (0.192 um) and the far background they are 26.6 and 9.5, all four values within tolerance, as they also
are at 26 to 44 pixels out; with the adopted pixel only the 14 to 20 pixel band (25.2 and 8.9) passes. With the
field proportional to the volts the slopes are 13.1 and 4.4 (0.267 um) or 18.0 and 6.0 (8-um pixel), and fail. The
local backgrounds break down on whole profiles (Pearson 0.012 to 0.137 with the adopted pixel, 0.005 and -0.02 with the
8-um pixel, where the subtracted background at some angles
approaches the trace intensity).

## fig3-fluovolt-nonlinear

> FluoVolt departed from linearity from about +-200 mV IMPC, while Di-8-ANEPPS stayed nearly linear from -500 to +500 mV; so FluoVolt's early departure reflects its dynamic range, not electroporation.

Criterion: smooths (LOESS 0.3, 0.5, 0.75; 50-mV bin means) against the line of fig3-dye-sensitivity; FluoVolt must
depart (smooth closer to zero by more than 10% of the line) at every |IMPC| from 220 mV to the end of the data on
both sides, and Di-8-ANEPPS must not depart anywhere from 150 to 500 mV, under every smooth.

In [ ]:
# fig3-fluovolt-nonlinear: smoothed dF against the line through the origin of fig3-dye-sensitivity, over the
# same points. Smooth readings: LOESS span 0.3, 0.5, 0.75 and means in 50-mV bins. A departure is the smooth
# lying closer to zero than the line by more than 10% of the line's value. FluoVolt must depart at every |IMPC|
# from 220 mV to the end of the data on both sides; Di-8-ANEPPS must not depart at any |IMPC| from 150 to 500 mV.
def smooths(x, y):
    o = np.argsort(x)
    x, y = x[o], y[o]
    out = {}
    for frac in (0.3, 0.5, 0.75):
        sm = lowess(y, x, frac=frac, it=0, delta=2.0, return_sorted=True)
        out[f"loess {frac}"] = (sm[:, 0], sm[:, 1])
    edges = np.arange(np.floor(x.min() / 50) * 50, x.max() + 50, 50)
    idx = np.digitize(x, edges)
    cx = np.array([x[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
    cy = np.array([y[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
    out["bins 50 mV"] = (cx, cy)
    return out


def departs(sx, sy, slope):
    line = slope / 100 * sx
    return (np.abs(sy) < np.abs(line) - 0.1 * np.abs(line)) & (np.sign(sy) == np.sign(line)) | (np.sign(sy) != np.sign(line))


for bg in strobe.BACKGROUNDS:
    res_bg = []
    for name in ("FluoVolt", "Di-8-ANEPPS"):
        x, y, slope = CAL[(name, bg, "radius_perimeter")]
        for k, (sx, sy) in smooths(x, y).items():
            dep = departs(sx, sy, slope)
            if name == "FluoVolt":
                m = np.abs(sx) >= 220
                ok = bool(dep[m].all()) if m.any() else False
                first = [f"{side}: {np.abs(sx[(np.sign(sx) == sg) & dep & (np.abs(sx) >= 100)]).min():.0f}"
                         if ((np.sign(sx) == sg) & dep & (np.abs(sx) >= 100)).any() else f"{side}: none"
                         for side, sg in (("neg", -1), ("pos", 1))]
                res_bg.append(ok)
                print(f"bg={bg:4s} FluoVolt {k:10s}: departs at every |IMPC|>=220 ({m.sum()} smooth points): {ok}; "
                      f"smallest |IMPC| >= 100 mV with a departure {first}; data to {sx.min():.0f}/{sx.max():.0f} mV")
            else:
                m = (np.abs(sx) >= 150) & (np.abs(sx) <= 500)
                nd = int(dep[m].sum())
                ok = nd == 0
                res_bg.append(ok)
                worst = (sy[m] / (slope / 100 * sx[m])).min() if m.any() else np.nan
                print(f"bg={bg:4s} Di-8     {k:10s}: smooth points 150-500 mV departing: {nd} of {m.sum()} -> no departure: {ok}; "
                      f"smallest smooth/line ratio {worst:.2f}")
    print(f"bg={bg}: all readings pass: {all(res_bg)}")

In [ ]:
# Variant (not the criterion): each side fit through the origin on its own (0 to +-150 mV), and the
# LOESS 0.5 smooth compared with that side's line.
for name in ("FluoVolt", "Di-8-ANEPPS"):
    x, y, slope = CAL[(name, "far", "radius_perimeter")]
    for side, sg in (("anodic (IMPC<0)", -1), ("cathodic (IMPC>0)", 1)):
        m = np.sign(x) == sg
        b = fit0(x[m], y[m])[0]
        sx, sy = smooths(x[m], y[m])["loess 0.5"]
        mm = (np.abs(sx) >= 150) & (np.abs(sx) <= 500)
        ratio = sy[mm] / (b / 100 * sx[mm])
        print(f"{name:12s} {side:18s} slope {b:5.2f} %/100 mV; LOESS 0.5 / own line over 150-500 mV: "
              f"{ratio.min():.2f} to {ratio.max():.2f}")

# Verification round 2: the same side fits without the 43 V/cm sets (whose Di-8-ANEPPS cathodic values are
# negative, fig2b-di8-bleaching), to test whether those points pull the cathodic fit down.
for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
    xs, ys = [], []
    for c in cells:
        for v in VOLTS[1:4]:
            s = set_of(c, v)
            xs.append(strobe.impc_profile(strobe.VOLT_TO_VCM[v], s["radius_perimeter"]))
            ys.append(strobe.collapsed_df(s, AXIS, "far"))
    x, y = np.concatenate(xs), np.concatenate(ys)
    for side, sg in (("anodic (IMPC<0)", -1), ("cathodic (IMPC>0)", 1)):
        m = np.sign(x) == sg
        b, _, _, n = fit0(x[m], y[m])
        sx, sy = smooths(x[m], y[m])["loess 0.5"]
        mm = (np.abs(sx) >= 150) & (np.abs(sx) <= 500)
        ratio = sy[mm] / (b / 100 * sx[mm])
        print(f"without 43 V/cm: {name:12s} {side:18s} slope {b:5.2f} %/100 mV ({n} points); "
              f"LOESS 0.5 / own line over 150-500 mV: {ratio.min():.2f} to {ratio.max():.2f}")

In [ ]:
# Variants (verification round 1): the 8-um pixel and the field proportional to the volts, each with its
# own line (the fig3-dye-sensitivity fit under the same variant); every smooth must pass, as in the criterion.
def nonlin(name, x, y, slope):
    res = {}
    for k, (sx, sy) in smooths(x, y).items():
        dep = departs(sx, sy, slope)
        if name == "FluoVolt":
            m = np.abs(sx) >= 220
            res[k] = bool(dep[m].all()) if m.any() else False
        else:
            m = (np.abs(sx) >= 150) & (np.abs(sx) <= 500)
            res[k] = (int(dep[m].sum()) == 0, round(float((sy[m] / (slope / 100 * sx[m])).min()), 2) if m.any() else np.nan)
    return res


for vlab, vmap, sc in (("8-um pixel", strobe.VOLT_TO_VCM, scale), ("proportional volts", strobe.VOLT_TO_VCM_PROP, 1.0)):
    for bg in strobe.BACKGROUNDS:
        line = []
        for name, cells in (("FluoVolt", FV_CELLS), ("Di-8-ANEPPS", D8_CELLS)):
            x, y = calib_points(cells, bg, "radius_perimeter", vmap, sc)
            sl = fit0(x, y)[0]
            line.append(f"{name} (slope {sl:.1f}): {nonlin(name, x, y, sl)}")
        print(f"{vlab}, bg={bg}:\n   " + "\n   ".join(line))

**Result: not reproduced.** Di-8-ANEPPS departs from the shared line on the positive (cathodic) side under every
smooth and background (far: 2709 of 5418 LOESS points from 150 to 500 mV; bins 5 of 14; smooth/line down to 0.73).
Much of it is an asymmetry between the poles: fit side by side, the anodic slope is 11.6 and the cathodic 3.1%
per 100 mV (far). There is also a bend: the anodic LOESS 0.5 smooth lies at 0.58 to 0.80 of its own line over 150 to
500 mV. Without the 43 V/cm sets, whose Di-8-ANEPPS cathodic values are negative (fig2b-di8-bleaching), the cathodic
slope rises to 4.6 and the anodic falls to 10.2% per 100 mV, so those points pull the cathodic fit down but account
for only part of the asymmetry; the anodic smooth then lies at 0.67 to 0.92 of its line. FluoVolt departs at every |IMPC| from 220 mV under LOESS 0.5 and 0.75 but not
under LOESS 0.3 or the bin means (far), so its part also fails the "every reading" condition. With the 8-um pixel the
FluoVolt part passes under every smooth and background, but Di-8-ANEPPS still departs under every smooth (smooth/line
down to 0.59 to 0.85); with the field proportional to the volts both parts fail.

In [ ]:
# Pole IMPC per cell and field strength: Eq. (1) averaged over the +-5 degree arcs (perimeter radius).
# Variants (verification round 1): equal-area radius (impc_area), the 8-um pixel (impc_8um) and the
# field proportional to the volts (impc_prop).
def pole_impc(subj, v, rkey="radius_perimeter", scale=1.0, vmap=strobe.VOLT_TO_VCM):
    s = set_of(subj, v)
    prof = strobe.impc_profile(vmap[v], s[rkey] * scale)
    return strobe.pole_values(prof)


CALCS = {"impc": {}, "impc_area": dict(rkey="radius_area"), "impc_8um": dict(scale=scale),
         "impc_prop": dict(vmap=strobe.VOLT_TO_VCM_PROP)}
rows = []
for c in FV_CELLS_ALT + D8_CELLS:
    for v in VOLTS:
        r = dict(subject=c, E=strobe.VOLT_TO_VCM[v])
        for k, kw in CALCS.items():
            r[f"{k}_c"], r[f"{k}_a"] = pole_impc(c, v, **kw)
        rows.append(r)
IM = pd.DataFrame(rows)
P = P.drop(columns=[c for c in IM.columns if c in P and c not in ("subject", "E")]).merge(IM, on=["subject", "E"])


def measured_vs_calc(cells, bg, sens, E, pole, paired, calc="impc"):
    d = P[(P.subject.isin(cells)) & (P.bg == bg) & (P.E == E)]
    meas = d["cath" if pole == "c" else "anod"] / sens * 100
    calc = d[f"{calc}_{pole}"]
    return meas.mean(), calc.mean(), ttest(meas, calc, paired)


## fig4a-fluovolt-limits

> FluoVolt measured and calculated IMP matched only at 43 and 150 V/cm; with stronger 1-us pulses the pole dF peaked at 71 +- 10% (cathodic) and -83.5 +- 5% (anodic), about +260 and -330 mV.

Criterion: maxima of the mean collapsed pole dF over field strengths (four FluoVolt cells) within 71 +- 10% and
-83.5 +- 5%; measured IMP (dF / 26%) against pole IMPC, Student's t-test paired and unpaired: no difference at 43
and 150 V/cm, measured below calculated at either pole at each strength from 230 V/cm.

In [ ]:
# fig4a-fluovolt-limits: maxima of mean collapsed pole dF over field strengths (4 FluoVolt cells); measured
# IMP (dF / 26%) against pole IMPC by Student's t-test, readings (a) paired, (b) unpaired. No difference at
# 43 and 150 V/cm at either pole; measured below calculated at either pole at each of 230 V/cm and above.
for bg in strobe.BACKGROUNDS:
    m, se = means(FV_CELLS, bg)
    mc, ma = m.cath.max(), m.anod.min()
    print(f"bg={bg:4s} max cathodic {mc:.1f}% at {m.cath.idxmax()} V/cm (71 +- 10: {abs(mc - 71) <= 10}); "
          f"max anodic {ma:.1f}% at {m.anod.idxmin()} V/cm (-83.5 +- 5: {abs(ma + 83.5) <= 5})")
    for paired in (True, False):
        rows = []
        for E in FIELDS:
            r = {"E": E}
            for pole in ("c", "a"):
                me, ca, p = measured_vs_calc(FV_CELLS, bg, 26, E, pole, paired)
                r[f"{pole}_meas"], r[f"{pole}_calc"], r[f"{pole}_p"] = round(me), round(ca), round(p, 4)
            r["c_below"] = r["c_p"] < 0.05 and abs(r["c_meas"]) < abs(r["c_calc"])
            r["a_below"] = r["a_p"] < 0.05 and abs(r["a_meas"]) < abs(r["a_calc"])
            rows.append(r)
        t = pd.DataFrame(rows).set_index("E")
        low_ok = bool((t.loc[[43, 150], ["c_p", "a_p"]] >= 0.05).all().all())
        high_ok = bool((t.loc[230:, "c_below"] | t.loc[230:, "a_below"]).all())
        print(f"   {'paired' if paired else 'unpaired'}: no difference at 43/150: {low_ok}; measured below calculated at "
              f"either pole at every E >= 230: {high_ok}")
        if bg == "far":
            print(t.to_string())

In [ ]:
# Variants (verification round 1). (1) The measured-against-calculated conditions under each IMPC variant
# (equal-area radius, 8-um pixel, field proportional to the volts). (2) The maxima and all conditions under
# every background reading (paired test, adopted IMPC).
def fig4a_mvc(bg, paired, calc="impc"):
    t = {(E, pl): measured_vs_calc(FV_CELLS, bg, 26, E, pl, paired, calc) for E in FIELDS for pl in "ca"}
    low_ok = all(t[(E, pl)][2] >= 0.05 for E in (43, 150) for pl in "ca")
    high_ok = all(any(t[(E, pl)][2] < 0.05 and abs(t[(E, pl)][0]) < abs(t[(E, pl)][1]) for pl in "ca")
                  for E in FIELDS if E >= 230)
    return low_ok, high_ok, t


for calc in CALCS:
    for bg in strobe.BACKGROUNDS:
        r = []
        for paired in (True, False):
            lo, hi, t = fig4a_mvc(bg, paired, calc)
            r.append(f"{'paired' if paired else 'unpaired'}: none at 43/150 {lo}, below at >=230 {hi}")
        _, _, t = fig4a_mvc(bg, True, calc)
        print(f"{calc:9s} bg={bg:4s} {'; '.join(r)}; 150 V/cm cath {t[(150, 'c')][0]:.0f} vs {t[(150, 'c')][1]:.0f} mV "
              f"(p {t[(150, 'c')][2]:.4f}), anod {t[(150, 'a')][0]:.0f} vs {t[(150, 'a')][1]:.0f} (p {t[(150, 'a')][2]:.4f})")


def _f4a(bg):
    m, _ = means(FV_CELLS, bg)
    lo, hi, _ = fig4a_mvc(bg, True)
    tol = abs(m.cath.max() - 71) <= 10 and abs(m.anod.min() + 83.5) <= 5
    return dict(max_cath=round(m.cath.max(), 1), at_c=m.cath.idxmax(), max_anod=round(m.anod.min(), 1), at_a=m.anod.idxmin(),
                maxima_in_tol=tol, none_at_43_150=lo, below_from_230=hi, all=tol and lo and hi)
print(sweep(_f4a).to_string())


# Verification round 2: measured against calculated IMP at 43 and 150 V/cm under every background reading
# (paired test, adopted IMPC), with the direction of each difference.
rows = []
for bg in ALL_BG:
    r = {}
    for E in (43, 150):
        for pl in "ca":
            me, ca, p = measured_vs_calc(FV_CELLS, bg, 26, E, pl, True)
            r[f"{E}_{pl}"] = f"{me:.0f} vs {ca:.0f} (p {p:.4f}, {'below' if abs(me) < abs(ca) else 'above'})"
    rows.append(pd.Series(r, name=bg))
print(pd.DataFrame(rows).to_string())

**Result: not reproduced.** Under the far background the maxima are 49.3% (cathodic, at 360 V/cm) and -72.1%
(anodic, at 780 V/cm), both outside tolerance; near gives 80.6% (within) and -89.1% (outside), none 23.6 and
-41.1%. In the sweep only the 14 to 20 pixel band puts both maxima within tolerance (61.2 and -79.5%), and there the
measured IMP is already below the calculated at the anodic pole at 150 V/cm (-191 against -205 mV, paired p =
0.019; cathodic 188 against 205 mV, p = 0.35). Under the far background measured IMP is below the
calculated value at every strength from 230 V/cm (paired p 0.0005 to 0.0041), but already at 150 V/cm (155 against
205 mV cathodic, paired p = 0.037; anodic p = 0.0045), so the match is found only at 43 V/cm. The equal-area radius
and the proportional field give the same; the 8-um pixel (162 mV calculated at 150 V/cm, paired p 0.65 and 0.25) and
the near background satisfy the measured-against-calculated conditions under the paired test, but neither brings both
maxima within tolerance.

## fig4a-fluovolt-cathodic-decline

> FluoVolt cathodic polarization fell from 71% at 360 V/cm to 45 to 50% at 1000 to 1400 V/cm.

Criterion: collapsed cathodic pole dF, mean of the four FluoVolt cells; 71 +- 10% at 360 V/cm and each of 1000, 1200
and 1400 V/cm within 40.5 to 55%.

In [ ]:
# fig4a-fluovolt-cathodic-decline: collapsed cathodic pole dF, mean of the 4 FluoVolt cells; 360 V/cm within
# 71 +- 10 and each of 1000, 1200, 1400 V/cm within 40.5-55%.
for cells, lab in ((FV_CELLS, "4 cells"), (FV_CELLS_ALT, "5 cells (+c5)")):
    for bg in strobe.BACKGROUNDS:
        m, se = means(FV_CELLS if lab == "4 cells" else FV_CELLS_ALT, bg)
        ok = abs(m.cath[360] - 71) <= 10 and all(40.5 <= m.cath[E] <= 55 for E in (1000, 1200, 1400))
        print(f"{lab}, bg={bg:4s}: cathodic 360 {m.cath[360]:.1f} (s.e. {se.cath[360]:.1f}); 1000/1200/1400: "
              f"{m.cath[1000]:.1f}/{m.cath[1200]:.1f}/{m.cath[1400]:.1f} -> {ok}")
m, _ = means(FV_CELLS, "far", cols=("cath_dark",))
print("variant, dark frames kept (4 cells, far): cathodic 360/1000/1200/1400", m.cath_dark[[360, 1000, 1200, 1400]].round(1).tolist())
print(P[(P.subject.isin(FV_CELLS)) & (P.bg == "far")].pivot(index="subject", columns="E", values="cath").round(1).to_string())

In [ ]:
# Variant (verification round 1): every background reading.
def _f4ad(bg):
    m, _ = means(FV_CELLS, bg)
    return dict(c360=round(m.cath[360], 1), c1000=round(m.cath[1000], 1), c1200=round(m.cath[1200], 1),
                c1400=round(m.cath[1400], 1), ok=abs(m.cath[360] - 71) <= 10 and all(40.5 <= m.cath[E] <= 55 for E in (1000, 1200, 1400)))
print(sweep(_f4ad).to_string())

**Result: not reproduced.** Far background: 49.3% at 360 V/cm, then 36.5, 38.3 and 40.9% at 1000, 1200 and 1400
V/cm (only the last within range). The decline itself is present (by 17 to 26% from 360 V/cm, the paper's 30 to
37%), but the values miss under every background reading (near 78.3 then 66.6 to 80.6; none 23.6 then 16.7 to 18.1).
The closest band of the sweep, 14 to 20 pixels, gives 61.2 then 48.2, 50.9 and 55.5%, failing only at 1400 V/cm (by
0.5 point). Per cell the 1400 V/cm value is above the 1000 V/cm value in three of four cells.

## fig4b-di8-limits

> With Di-8-ANEPPS and 1-us pulses, measured IMP fell below the calculated value at 570 V/cm, where dF peaked at 57 +- 14% (cathodic) and -55 +- 4% (anodic), over 600 mV.

Criterion: the largest mean |dF| at each pole at 570 V/cm and within tolerance (57 +- 14%, -55 +- 4%); measured IMP
(dF / 9.1%) below pole IMPC at 570 V/cm (either pole, p < 0.05) but not below it at 43 to 360 V/cm.

In [ ]:
# fig4b-di8-limits: largest mean |dF| at each pole at 570 V/cm and within tolerance (57 +- 14; -55 +- 4);
# measured IMP (dF / 9.1%) below pole IMPC at 570 V/cm (either pole, p < 0.05) but not below it at 43-360.
def fig4b_test(cells, bg, paired, calc="impc"):
    m, _ = means(cells, bg)
    c1 = m.cath.idxmax() == 570 and m.anod.idxmin() == 570
    c2 = abs(m.cath.max() - 57) <= 14 and abs(m.anod.min() + 55) <= 4
    below570 = False
    notbelow_low = True
    for pole in ("c", "a"):
        me, ca, p = measured_vs_calc(cells, bg, 9.1, 570, pole, paired, calc)
        below570 |= (p < 0.05 and abs(me) < abs(ca))
        for E in (43, 150, 230, 360):
            me, ca, p = measured_vs_calc(cells, bg, 9.1, E, pole, paired, calc)
            if p < 0.05 and abs(me) < abs(ca):
                notbelow_low = False
    return dict(max_at_570=c1, in_tol=c2, below_at_570=below570, not_below_43_360=notbelow_low), m


for bg in strobe.BACKGROUNDS:
    for paired in (True, False):
        r, m = fig4b_test(D8_CELLS, bg, paired)
        print(f"bg={bg:4s} {'paired  ' if paired else 'unpaired'}: max cath {m.cath.max():.1f} at {m.cath.idxmax()}, "
              f"max anod {m.anod.min():.1f} at {m.anod.idxmin()} | {r} -> {all(r.values())}; "
              f"5-cell subsets passing {subset_share(lambda c: all(fig4b_test(c, bg, paired)[0].values()))}")
    rows = []
    for E in FIELDS:
        r = {"E": E}
        for pole in ("c", "a"):
            me, ca, p = measured_vs_calc(D8_CELLS, bg, 9.1, E, pole, True)
            r[f"{pole}_meas_mV"], r[f"{pole}_calc_mV"], r[f"{pole}_p_paired"] = round(me), round(ca), round(p, 4)
            r[f"{pole}_p_unpaired"] = round(measured_vs_calc(D8_CELLS, bg, 9.1, E, pole, False)[2], 4)
        rows.append(r)
    print(pd.DataFrame(rows).set_index("E").to_string())

In [ ]:
# Variants (verification round 1). (1) Each IMPC variant (equal-area radius, 8-um pixel, field proportional
# to the volts), far background. (2) Every background reading (adopted IMPC, paired and unpaired).
for calc in CALCS:
    for paired in (True, False):
        r, m = fig4b_test(D8_CELLS, "far", paired, calc)
        print(f"{calc:9s} far {'paired  ' if paired else 'unpaired'}: {r}")


def _f4b(bg):
    out = {}
    for paired in (True, False):
        r, m = fig4b_test(D8_CELLS, bg, paired)
        out[f"all_{'paired' if paired else 'unpaired'}"] = all(r.values())
    return dict(max_cath=round(m.cath.max(), 1), at_c=m.cath.idxmax(), max_anod=round(m.anod.min(), 1), at_a=m.anod.idxmin(),
                **{k: v for k, v in r.items()}, **out)
print(sweep(_f4b).to_string())

**Result: not reproduced.** Far background: the cathodic maximum is at 570 V/cm (41.7%, 1.3 points below the
tolerance) but the anodic maximum is at 780 V/cm (-41.4%, against -40.4% at 570) and outside tolerance; measured IMP
is below IMPC at 570 V/cm (paired p = 0.0035 cathodic, < 0.0001 anodic), but also at 360 V/cm anodic (p = 0.0012)
and at 43 V/cm, where the cathodic dF is negative from bleaching. The equal-area radius, the 8-um pixel and the
proportional field give the same four results. No background reading, no band of the sweep and none of the 56
five-cell subsets meets every condition. The near background puts both maxima at 570 V/cm (67.9 and -61.4%, the
anodic outside tolerance), but then measured IMP is not below IMPC at 570 V/cm (p 0.70 and 0.23).

## fig4b-first-10-pulses

> Averaged over only the first 10 of the 60 pulses, Di-8-ANEPPS cathodic polarization reached 66 +- 15% (710 +- 200 mV), so the 600-mV maximum is not the dye's limit.

Criterion: polarization image from the first 10 of the 60 pulse images; the largest mean cathodic dF over field
strengths within 66 +- 15% and above the 60-image maximum of fig4b-di8-limits.

In [ ]:
# fig4b-first-10-pulses: polarization image collapsed from the first 10 of the 60 pulse images; largest mean
# cathodic dF over field strengths within 66 +- 15 and above the 60-image maximum of fig4b-di8-limits.
for bg in strobe.BACKGROUNDS:
    m10, se10 = means(D8_CELLS, bg, cols=("cath10", "anod10"))
    m60, _ = means(D8_CELLS, bg)
    ok = abs(m10.cath10.max() - 66) <= 15 and m10.cath10.max() > m60.cath.max()
    print(f"bg={bg:4s}: first-10 cathodic max {m10.cath10.max():.1f}% at {m10.cath10.idxmax()} V/cm "
          f"(s.e. {se10.cath10[m10.cath10.idxmax()]:.1f}); 60-image max {m60.cath.max():.1f}% -> {ok}; "
          f"5-cell subsets passing {subset_share(lambda c: abs(means(c, bg, ('cath10',))[0].cath10.max() - 66) <= 15 and means(c, bg, ('cath10',))[0].cath10.max() > means(c, bg)[0].cath.max())}")
    print(f"{'':10s} first-10 cathodic by field: {m10.cath10.round(1).to_dict()}")

In [ ]:
# Variant (verification round 1): every background reading.
def _f10(bg):
    m10, se10 = means(D8_CELLS, bg, cols=("cath10",))
    m60, _ = means(D8_CELLS, bg)
    return dict(max10=round(m10.cath10.max(), 1), at=m10.cath10.idxmax(), se=round(se10.cath10[m10.cath10.idxmax()], 1),
                max60=round(m60.cath.max(), 1), ok=abs(m10.cath10.max() - 66) <= 15 and m10.cath10.max() > m60.cath.max())
print(sweep(_f10).to_string())

**Result: not reproduced.** Far background: 43.1% at 570 V/cm, above the 60-image maximum (41.7%) but below the
tolerance (51 to 81%); 5 of 56 five-cell subsets pass. Without background 16.5%. The near background (69.8% at 1000
V/cm, s.e. 16.1; 43 of 56 subsets) and the 14 to 20 pixel band (53.3% at 570 V/cm) pass; every band from 20 pixels
out fails (41.0 to 47.0%).

## fig4b-cathodic-asymmetry

> Above 780 V/cm the pole IMP fell progressively with field strength, more at the cathodic pole; at 1400 V/cm the cathodic pole no longer charged and the anodic pole fell to about -30% dF (-330 mV).

Criterion: (1) mean |dF| at each pole lower at 1000, 1200 and 1400 than at 780 V/cm; (2) a larger fraction lost from
780 to 1400 V/cm at the cathodic pole; (3) cathodic dF at 1400 V/cm not different from zero (one-sample t-test);
(4) anodic dF at 1400 V/cm within -30 +- 3%.

In [ ]:
# fig4b-cathodic-asymmetry: (1) mean |dF| at each pole lower at 1000, 1200 and 1400 than at 780 V/cm;
# (2) fraction lost 780 -> 1400 larger at the cathodic pole; (3) cathodic dF at 1400 V/cm not different from
# zero (one-sample t-test, p >= 0.05); (4) anodic dF at 1400 V/cm within -30 +- 3.
def asym_test(cells, bg):
    m, _ = means(cells, bg)
    c, a = m.cath.abs(), m.anod.abs()
    d = P[(P.subject.isin(cells)) & (P.bg == bg) & (P.E == 1400)]
    p0 = stats.ttest_1samp(d.cath, 0).pvalue
    lc, la = 1 - c[1400] / c[780], 1 - a[1400] / a[780]
    conds = dict(decrease_from_780=bool(all(c[E] < c[780] and a[E] < a[780] for E in (1000, 1200, 1400))),
                 cath_loses_more=bool(lc > la), cath1400_not_diff_0=bool(p0 >= 0.05), anod1400_in_tol=bool(abs(m.anod[1400] + 30) <= 3))
    return conds, (c, a, lc, la, p0, m.cath[1400], m.anod[1400])


for bg in strobe.BACKGROUNDS:
    conds, (c, a, lc, la, p0, c14, a14) = asym_test(D8_CELLS, bg)
    print(f"bg={bg:4s}: |cath| 780-1400 {c[[780, 1000, 1200, 1400]].round(1).tolist()}, |anod| {a[[780, 1000, 1200, 1400]].round(1).tolist()}; "
          f"lost cath {lc:.2f}, anod {la:.2f}; cath 1400 {c14:.1f}% (p vs 0 = {p0:.3f}); anod 1400 {a14:.1f}%")
    print(f"{'':10s} {conds} -> {all(conds.values())}; 5-cell subsets passing {subset_share(lambda cc: all(asym_test(cc, bg)[0].values()))}")

In [ ]:
# Variant (verification round 1): every background reading.
def _asy(bg):
    conds, (c, a, lc, la, p0, c14, a14) = asym_test(D8_CELLS, bg)
    return dict(anod1400=round(a14, 1), cath1400=round(c14, 1), p0=round(p0, 3), **conds, all=all(conds.values()))
print(sweep(_asy).to_string())

**Result: partially reproduced.** Under the far background conditions 1 to 3 hold: both poles fall from 780 V/cm
(cathodic 38.6, 31.4, 19.8, 7.5%; anodic 41.4, 36.9, 30.8, 21.6%), the cathodic pole loses more (81 against 48%),
and the cathodic value at 1400 V/cm does not differ from zero (7.5%, p = 0.29). The anodic value at 1400 V/cm is
-21.6%, outside -30 +- 3% (none -10.7%; every band from 14 pixels out -21.3 to -23.8%). The near background meets all
four conditions (anodic -27.7%, cathodic 21.2%, p = 0.12; 15 of 56 subsets), so the fourth condition turns on the
background distance.

## fig4c-equator-follows-cosine

> Membrane regions nearer the equator were unaffected by pole electroporation and followed the cosine dependence of the Schwan equation up to 1000 V/cm.

Claim of absence. Measured IMP and IMPC averaged per cell in arcs beside the two equator points, (a) 15 or (b) 30
degrees wide, at 570, 780 and 1000 V/cm; Student's t-test, (c) paired and (d) unpaired. Reproduced when no arc
differs under (c) and (d); not reproduced when any differs under both.

In [ ]:
# fig4c-equator-follows-cosine (claim of absence): measured IMP (dF / 9.1%) and IMPC per perimeter point,
# averaged per cell within arcs beside the two equator points (90 and 270 degrees from the cathodic pole),
# on each side: readings (a) 15 and (b) 30 degrees wide. At 570, 780 and 1000 V/cm, Student's t-test
# (c) paired, (d) unpaired, p < 0.05, over the Di-8-ANEPPS cells.
def eq_arcs(width):
    th = np.arange(strobe.N_POINTS)
    return {"90 cath side": (th >= 90 - width) & (th < 90), "90 anod side": (th > 90) & (th <= 90 + width),
            "270 anod side": (th >= 270 - width) & (th < 270), "270 cath side": (th > 270) & (th <= 270 + width)}


def fig4c_rows(cells, bg, scale=1.0, vmap=strobe.VOLT_TO_VCM):
    rows = []
    for width, wl in ((15, "a"), (30, "b")):
        for E in (570, 780, 1000):
            v = [k for k, e in strobe.VOLT_TO_VCM.items() if e == E][0]
            for name, m in eq_arcs(width).items():
                meas, calc = [], []
                for c in cells:
                    s = set_of(c, v)
                    meas.append(strobe.collapsed_df(s, AXIS, bg)[m].mean() / 9.1 * 100)
                    calc.append(strobe.impc_profile(vmap[v], s["radius_perimeter"] * scale)[m].mean())
                rows.append(dict(width=wl, E=E, arc=name, meas=np.mean(meas), calc=np.mean(calc),
                                 ratio=np.mean(meas) / np.mean(calc), p_paired=ttest(meas, calc, True),
                                 p_unpaired=ttest(meas, calc, False)))
    return pd.DataFrame(rows)


for bg in strobe.BACKGROUNDS:
    t = fig4c_rows(D8_CELLS, bg)
    nc, nd = int((t.p_paired < 0.05).sum()), int((t.p_unpaired < 0.05).sum())
    both = int(((t.p_paired < 0.05) & (t.p_unpaired < 0.05)).sum())
    print(f"bg={bg:4s}: arcs differing (of {len(t)}): paired {nc}, unpaired {nd}, under both {both}")
    if bg == "far":
        print(t.round(3).to_string())
    else:
        print("   ratio measured/calculated range", round(t.ratio.min(), 2), "to", round(t.ratio.max(), 2))
t8 = fig4c_rows(D8_CELLS, "far", scale)
print(f"8-um pixel variant (far): arcs differing paired {(t8.p_paired < 0.05).sum()}, unpaired {(t8.p_unpaired < 0.05).sum()}; "
      f"ratio {t8.ratio.min():.2f} to {t8.ratio.max():.2f}")

In [ ]:
# Variants (verification round 1): the field proportional to the volts; the share of the 56 five-cell
# subsets with no arc differing under both tests (far background); every background reading.
for bg in strobe.BACKGROUNDS:
    t = fig4c_rows(D8_CELLS, bg, vmap=strobe.VOLT_TO_VCM_PROP)
    print(f"proportional volts, bg={bg:4s}: arcs differing paired {(t.p_paired < 0.05).sum()}, unpaired "
          f"{(t.p_unpaired < 0.05).sum()}, both {((t.p_paired < 0.05) & (t.p_unpaired < 0.05)).sum()}; ratio {t.ratio.min():.2f} to {t.ratio.max():.2f}")
print("far, 5-cell subsets with no arc differing under both tests:",
      subset_share(lambda c: not ((fig4c_rows(c, "far").p_paired < 0.05) & (fig4c_rows(c, "far").p_unpaired < 0.05)).any()))


def _f4c(bg):
    t = fig4c_rows(D8_CELLS, bg)
    return dict(paired=int((t.p_paired < 0.05).sum()), unpaired=int((t.p_unpaired < 0.05).sum()),
                both=int(((t.p_paired < 0.05) & (t.p_unpaired < 0.05)).sum()), ratio_min=round(t.ratio.min(), 2), ratio_max=round(t.ratio.max(), 2))
print(sweep(_f4c).to_string())

**Result: not reproduced.** Under the far background 16 of 24 arcs differ under both tests (ratios of measured to
calculated 0.40 to 0.98); the arc on the cathodic side of the 270-degree equator is low at every strength (ratio
0.40 to 0.61). None of the 56 five-cell subsets is free of differing arcs. The near background leaves one arc
differing under both tests, no background all 24; in the sweep from 14 pixels out 6 to 16 arcs differ; with the 8-um
pixel 7 (paired) and 9 (unpaired) differ; with the field proportional to the volts 16. At 570 and 780 V/cm the
15-degree arcs on the anodic side and beside 90 degrees match (ratios 0.75 to 0.98, p 0.08 to 0.93).

## fig5b-conductivity-calibration

> By Eq. (2), the effective membrane conductivity barely affects charging while it stays far below cytosol and medium conductivities; a 10-fold rise reduces IMP by about 2% and a 100-fold rise by 12%.

Criterion: Eq. (2) with the constants of Methods 2.3; IMP reduction for a 10-fold and a 100-fold rise from 3e-7
S/m, readings (a) 1 - f/1.5 and (b) 1 - f(lambda)/f(3e-7). Both within 10% of 2% and 12%. This uses no recordings.

In [ ]:
# fig5b-conductivity-calibration: Eq. (2) with the constants of Methods 2.3; IMP reduction for a 10-fold and a
# 100-fold rise of the membrane conductivity from 3e-7 S/m. Readings (a) 1 - f/1.5, (b) 1 - f(lam)/f(3e-7).
f0 = strobe.f_factor(3e-7)
print(f"f(3e-7 S/m) = {f0:.5f}")
for fold, exp in ((10, 2), (100, 12)):
    f = strobe.f_factor(3e-7 * fold)
    ra, rb = (1 - f / 1.5) * 100, (1 - f / f0) * 100
    print(f"{fold:4d}-fold: f = {f:.5f}; reduction (a) {ra:.2f}%, (b) {rb:.2f}% (expected {exp} +- {0.1 * exp:.1f}: "
          f"(a) {abs(ra - exp) <= 0.1 * exp}, (b) {abs(rb - exp) <= 0.1 * exp})")
lam = np.logspace(-7, -2, 11)
print("curve (lambda_m S/m, % of 1.5):", [(f"{l:.1e}", round(strobe.f_factor(l) / 1.5 * 100, 2)) for l in lam])
for target in (2, 12):
    from scipy.optimize import brentq
    l = brentq(lambda x: (1 - strobe.f_factor(x) / 1.5) * 100 - target, 3e-7, 1)
    print(f"fold rise giving a {target}% reduction (reading a): {l / 3e-7:.1f}")

**Result: partially reproduced.** A 100-fold rise reduces the IMP by 11.87% (a) or 11.75% (b), as stated; a 10-fold
rise by 1.33% (a) or 1.20% (b), outside 2 +- 0.2%. Under Eq. (2) a 2% reduction needs a 15-fold rise. Eq. (2) gives a
smaller reduction at 10-fold than the paper states (1.33% against 2%), so its curve lies about 0.7 point above the
paper's value there; whether the paper's "~2%" is a rounding of a read-off value
from Fig. 5B cannot be told here.

## fig7b-charging-during-pulse

> Normalized to 150 V/cm, pole polarization during the conditioning pulse fell 1.5 times below linear scaling at 560 V/cm and 2-fold at 760 V/cm, and further at stronger fields (p < 0.01 for every point from 560 V/cm).

Criterion: the Fig. 4B data (Di-8-ANEPPS), pole dF as % of the 150 V/cm value against the line 100 x E / 150; ratio
of line to measured within 1.5 +- 0.15 at 570 and 2 +- 0.2 at 780 V/cm, above 2 at 1000 to 1400 V/cm, and every
point from 570 V/cm below the line at p < 0.01 (one-sample t-test of per-cell values). Readings: normalisation per
cell or of the means; each pole or the mean of the poles.

In [ ]:
# fig7b-charging-during-pulse: Fig. 4B data (Di-8-ANEPPS), pole dF as % of the 150 V/cm value against the line
# 100 x E / 150. Readings: normalisation (a) per cell, (b) of the means; pole (c) each pole, (d) mean of the poles.
# Ratio = line / measured. One-sample t-test of the per-cell % values against the line, p < 0.01 for every
# datapoint from 560 V/cm.
def fig7b(cells, bg, vmap=strobe.VOLT_TO_VCM):
    d = P[(P.subject.isin(cells)) & (P.bg == bg)].copy()
    d["mean_pole"] = (d.cath - d.anod) / 2
    out = []
    for pole in ("cath", "anod", "mean_pole"):
        w = d.pivot(index="subject", columns="E", values=pole).abs()
        pc = w.div(w[150], axis=0) * 100              # per cell
        pm = w.mean() / w.mean()[150] * 100            # of the means
        line = 100 * np.array([vmap[v] for v in VOLTS]) / vmap[50]   # the paper's 100 x E / 150 under the adopted map
        lineE = dict(zip(FIELDS, line))
        for norm, vals in (("per cell", pc.mean()), ("of means", pm)):
            ratio = pd.Series(line, index=FIELDS) / vals
            pv = {E: stats.ttest_1samp(pc[E], lineE[E]).pvalue for E in FIELDS if E >= 570}
            below = {E: bool(pc[E].mean() < lineE[E] and pv[E] < 0.01) for E in pv}
            ok = (abs(ratio[570] - 1.5) <= 0.15 and abs(ratio[780] - 2) <= 0.2 and all(ratio[E] > 2 for E in (1000, 1200, 1400))
                  and all(below.values()))
            out.append(dict(pole=pole, norm=norm, r570=ratio[570], r780=ratio[780], r1000=ratio[1000], r1200=ratio[1200],
                            r1400=ratio[1400], max_p=max(pv.values()), all_below_p01=all(below.values()), ok=ok))
    return pd.DataFrame(out)


for bg in strobe.BACKGROUNDS:
    t = fig7b(D8_CELLS, bg)
    print(f"bg={bg}:")
    print(t.round(3).to_string())
    print(f"   reproduced under some reading: {t.ok.any()}; 5-cell subsets passing {subset_share(lambda c: fig7b(c, bg).ok.any())}")

In [ ]:
# Variants (verification round 1). (1) Field proportional to the volts: the line is 100% x volts / 50 V
# (300% at the 150 V set in place of 380%). (2) Every background reading, adopted map.
for bg in strobe.BACKGROUNDS:
    t = fig7b(D8_CELLS, bg, strobe.VOLT_TO_VCM_PROP)
    print(f"proportional volts, bg={bg}:")
    print(t.round(3).to_string())
    print(f"   reproduced under some reading: {t.ok.any()}")
print(sweep(lambda bg: dict(any_reading_ok=bool(fig7b(D8_CELLS, bg).ok.any()),
                            r570_r780_first_reading=tuple(fig7b(D8_CELLS, bg).iloc[0][["r570", "r780"]].round(2)))).to_string())

**Result: reproduced, on the in-order matching of volts to V/cm.** Under the first reading (per cell, cathodic pole,
far background) the ratios are 1.47 at 570 and 2.15 at 780 V/cm, 3.4 to 13.3 at 1000 to 1400 V/cm, and every point
from 570 V/cm is below the line (largest p 0.001). All six readings pass under the far background (ratios 1.45 to
1.64 and 2.09 to 2.15), at least four of six under the near and no-background readings, and some reading passes at
every band of the sweep and with the far local background (not the near local one); 43 of 56 five-cell subsets pass (far). The result depends on the volts mapping: with the
field proportional to the volts the line at the 150 V set is 300% in place of 380%, the ratios fall to 1.15 to 1.29
at 570 and 1.61 to 1.66 at 780 V/cm, and no reading passes under any background. The data check (25 to 50 V triples
the response, against 2.0 for proportional volts) supports the in-order matching.

## Blocked claims

Fifteen claims need recordings that the dandiset does not hold (data check: every set is a 1-us pulse with
FluoVolt or Di-8-ANEPPS in standard solution, in escalations of the pulse amplitude):

- `fig2cd-stable-before-electroporation`, `fig2c-1000-within-set`, `fig2d-lasting-reduction`: the alternating
  150/230 V/cm sets followed by 1000 V/cm (Fig. 2C/D, n = 8). No cell in the dandiset has an alternating sequence.
- `fig4d-short-pulses-same-limit`: 0.3-us pulses imaged at 275 ns (Fig. 4D). Every file states a 1-us pulse.
- `fig5a-gd-extends-anodic`, `fig5a-gd-linear`, `fig5a-gd-cathodic`, `fig5a-gd-differences`, and
  `fig5c-gd-reduces-conductivity` (which also uses them): the 1 mM Gd3+ cells and their parallel controls (Fig. 5A).
  No session description mentions Gd3+, and there is no second group of escalations recorded in parallel.
- `fig6a-ca-single-pulse`, `fig6b-ca-train-responders`, `fig6b-ca-650`: Fluo-4 Ca2+ imaging (Fig. 6). The dandiset
  holds strobe images only.
- `fig7ab-test-pulse-recovery`, `fig7c-fast-conductivity-drop`, `fig7c-full-reseal-mild`: the two-pulse protocol with
  a 150 V/cm test pulse (Fig. 7A, and the lower panel of Fig. 7B). Every file states a single pulse.

The paper says its data are available on request. The only other dandiset that names the paper, 001518, is by its
title and description a comparison of FluoVolt and Di-8-ANEPPS responses at varying field strengths, not these
experiments; its files were not read here.

## Variant: Di-8-ANEPPS outlines on the outer membrane (verification round 2)

The overlays of the data check show several Di-8-ANEPPS outlines (c7, c10, c12, c13, c14) wavy and cutting inside
the membrane where bright bodies inside the cell lie close to it. Here every used Di-8-ANEPPS set is traced again with
`strobe.membrane_line_outer`: on each ray the outermost intensity peak that reaches 80% of the ray's range (in place of
the highest peak), median-filtered over 15 degrees and kept to the first six angular harmonics. The search is held near
each cell's median centre and radius, as in the re-tracing of the loading cell. The four Di-8-ANEPPS claims whose values
the outlines can move are run again under the three background readings: fig4b-di8-limits, fig4b-first-10-pulses,
fig4b-cathodic-asymmetry and fig4c-equator-follows-cosine. This is a variant; the outcomes stay on the adopted outline.

In [ ]:
# Re-trace the 72 used Di-8-ANEPPS sets with the outer-ridge outline (about 2 s per set).
t0 = time.time()
d8_keys = [(c, v) for c in D8_CELLS for v in VOLTS]
d8_rows = {(c, v): next(r for r in use.itertuples(index=False) if r.asset_id == set_of(c, v)["asset_id"]) for c, v in d8_keys}


def _extract_outer(key):
    row = d8_rows[key]
    k = next(i for i, s in enumerate(S) if s["asset_id"] == row.asset_id)
    s = strobe.extract_set(row.asset_id, hint=tuple(med.loc[k, ["cy", "cx", "r"]]), outline="outer")
    s.update(row._asdict())
    return k, s


with ThreadPoolExecutor(4) as ex:
    OUTER = dict(ex.map(_extract_outer, d8_keys))
print(f"{len(OUTER)} sets re-traced in {time.time() - t0:.0f} s")
ch = pd.DataFrame([dict(subject=s["subject"], volts=s["volts"], R_adopted=S[k]["radius_perimeter"], R_outer=s["radius_perimeter"],
                        sd_adopted=float(S[k]["r"].std()), sd_outer=float(s["r"].std()),
                        shift_px=float(np.abs(s["r"] - S[k]["r"]).mean()))
                   for k, s in OUTER.items()])
print("perimeter radius (um), outline roughness (s.d. of radius, px) and mean change of the radius at each angle (px), by cell:")
print(ch.groupby("subject")[["R_adopted", "R_outer", "sd_adopted", "sd_outer", "shift_px"]].mean().round(2).to_string())

# overlays: adopted outline (red) and outer-ridge outline (yellow), 150 and 350 V sets
fig, axs = plt.subplots(2, len(D8_CELLS), figsize=(2.0 * len(D8_CELLS), 4.4))
ang = np.deg2rad(np.arange(strobe.N_POINTS))
for j, subj in enumerate(D8_CELLS):
    for i, v in enumerate((150, 350)):
        k = next(k for k, s in OUTER.items() if s["subject"] == subj and s["volts"] == v)
        a, b = S[k], OUTER[k]
        ax = axs[i, j]
        lo, hi = np.percentile(a["crop"], [1, 99.5])
        ax.imshow(a["crop"], cmap="gray", vmin=lo, vmax=hi)
        y0, x0 = a["crop_origin"]
        for s, col in ((a, "r"), (b, "y")):
            ax.plot(s["cx"] + s["r"] * np.cos(ang) - x0, s["cy"] - s["r"] * np.sin(ang) - y0, col, lw=0.6)
        ax.set_xlim(0, 2 * strobe.CROP); ax.set_ylim(2 * strobe.CROP, 0); ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(subj.split("-")[-1], fontsize=8)
        if j == 0:
            ax.set_ylabel(f"{v} V", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# The four claims under the outer-ridge outline: the shared tables are rebuilt for the Di-8-ANEPPS cells with the
# re-traced sets, the claim functions run on them, and the adopted tables are then restored.
S_ADOPTED, P_ADOPTED = S, P
S = list(S_ADOPTED)
for k, s in OUTER.items():
    S[k] = s
rows = []
for subj in D8_CELLS:
    for v in VOLTS:
        s = set_of(subj, v)
        for bg in strobe.BACKGROUNDS:
            c, a = strobe.pole_values(strobe.collapsed_df(s, AXIS, bg))
            c10, a10 = strobe.pole_values(strobe.collapsed_df(s, AXIS, bg, pulse_frames=slice(0, 10)))
            r = dict(subject=subj, dye=s["dye"], volts=v, E=strobe.VOLT_TO_VCM[v], bg=bg, cath=c, anod=a, cath10=c10, anod10=a10,
                     R_area=s["radius_area"], R_perim=s["radius_perimeter"])
            for kk, kw in CALCS.items():
                r[f"{kk}_c"], r[f"{kk}_a"] = pole_impc(subj, v, **kw)
            rows.append(r)
P = pd.DataFrame(rows)
try:
    for bg in strobe.BACKGROUNDS:
        r4b = {f"{'paired' if pr else 'unpaired'}": fig4b_test(D8_CELLS, bg, pr)[0] for pr in (True, False)}
        m = fig4b_test(D8_CELLS, bg, True)[1]
        print(f"bg={bg:4s} fig4b-di8-limits: max cath {m.cath.max():.1f} at {m.cath.idxmax()}, max anod {m.anod.min():.1f} at "
              f"{m.anod.idxmin()}; paired {r4b['paired']}; all (paired / unpaired) {all(r4b['paired'].values())} / {all(r4b['unpaired'].values())}")
        m10, se10 = means(D8_CELLS, bg, cols=("cath10",))
        ok10 = abs(m10.cath10.max() - 66) <= 15 and m10.cath10.max() > m.cath.max()
        print(f"{'':9s} fig4b-first-10-pulses: first-10 cathodic max {m10.cath10.max():.1f}% at {m10.cath10.idxmax()} V/cm "
              f"(60-image max {m.cath.max():.1f}) -> {ok10}")
        conds, (c, a, lc, la, p0, c14, a14) = asym_test(D8_CELLS, bg)
        print(f"{'':9s} fig4b-cathodic-asymmetry: lost cath {lc:.2f}, anod {la:.2f}; cath 1400 {c14:.1f}% (p vs 0 {p0:.3f}); "
              f"anod 1400 {a14:.1f}%; {conds} -> {all(conds.values())}")
        t = fig4c_rows(D8_CELLS, bg)
        both = int(((t.p_paired < 0.05) & (t.p_unpaired < 0.05)).sum())
        print(f"{'':9s} fig4c-equator-follows-cosine: arcs differing (of {len(t)}) paired {(t.p_paired < 0.05).sum()}, "
              f"unpaired {(t.p_unpaired < 0.05).sum()}, both {both}; ratio {t.ratio.min():.2f} to {t.ratio.max():.2f}")
finally:
    S, P = S_ADOPTED, P_ADOPTED


**Variant result.** The outer-ridge outline lies on the outer membrane in the overlays where the adopted one cut
inside it (c7, c10, c12, c13, c14), and is smoother (s.d. of the radius 0.99 to 1.90 against 1.18 to 2.25 pixels by cell);
radii change by 0.35 um or less (perimeter radius by cell 9.86 to 12.72 um against 9.86 to 12.85). No outcome changes
under the adopted far background. fig4b-di8-limits: cathodic maximum 44.2% at 570 V/cm, anodic -43.0% at 780 V/cm
(outside -55 +- 4), and measured IMP below IMPC at 43 to 360 V/cm, as before. fig4b-first-10-pulses: 45.5% at 570 V/cm,
below 51 to 81% (adopted 43.1%). fig4b-cathodic-asymmetry: conditions 1 to 3 hold, anodic 1400 V/cm -23.1% against
-30 +- 3 (adopted -21.6%). fig4c-equator-follows-cosine: 16 of 24 arcs differ under both tests, as before. Under the
near background fig4b-first-10-pulses (72.7%) and fig4b-cathodic-asymmetry (anodic -28.2%) pass, as they do with the
adopted outline, and fig4b-di8-limits still fails (anodic maximum at 780 V/cm). The outlines move the Di-8-ANEPPS
values by a few points; the background distance remains what the outcomes turn on.


## Summary

| Claim | Outcome |
|---|---|
| `fig1-pole-dimming` | reproduced |
| `fig2a-fluovolt-saturation` | not reproduced |
| `fig2a-fluovolt-stable` | not reproduced |
| `fig2b-di8-climbs-to-570` | reproduced |
| `fig2b-di8-bleaching` | reproduced |
| `fig2b-di8-decline-within-set` | reproduced |
| `fig3-dye-sensitivity` | partially reproduced |
| `fig3-fluovolt-nonlinear` | not reproduced |
| `fig4a-fluovolt-limits` | not reproduced |
| `fig4a-fluovolt-cathodic-decline` | not reproduced |
| `fig4b-di8-limits` | not reproduced |
| `fig4b-first-10-pulses` | not reproduced |
| `fig4b-cathodic-asymmetry` | partially reproduced |
| `fig4c-equator-follows-cosine` | not reproduced |
| `fig5b-conductivity-calibration` | partially reproduced |
| `fig7b-charging-during-pulse` | reproduced |
| 15 claims on Fig. 2C/D, 4D, 5A, 5C, 6 and 7A/C | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

The shapes of the field-strength series reproduce: Di-8-ANEPPS polarization rises to 570 V/cm and falls beyond 780
V/cm, more at the cathodic pole, with the poles dimming at strong fields (fig1-pole-dimming), though under the adopted
background 16 of the 24 equatorial arcs tested also fall below the calculated IMP, the 270-degree cathodic-side arc
already at 570 V/cm (ratio 0.52); responses decline within the electroporating sets from 1000 V/cm, while at 780 V/cm
the decline cannot be told from bleaching (net cathodic change -3.3% against -2.6% at 570 V/cm, reading b); and polarization falls below linear scaling by the paper's factors at 570 and 780
V/cm. Most failures are of absolute dF values and of slopes against the calculated IMP. These rest on details the
files and the paper do not fix: where the background is measured (the three readings move the values by a factor of
three), the pixel size (which scales the calculated IMP), which of the cells the paper used, and how the generator
volts in the files map to the paper's V/cm. The sweep of the background distance added after verification round 1
shows that a band 4 to 5 um outside the membrane (14 to 20 pixels), where the membrane's glow is still about a sixth
of its excess, would bring the FluoVolt saturation values, both dye slopes and the first-10 Di-8-ANEPPS maximum within
tolerance, and the near band would satisfy fig4b-cathodic-asymmetry; the adopted band, 8 to 10 um out, is kept because
it is the one that is free of the cell's light, and the choice is not made to fit the paper's values. The in-order
matching of volts is supported by the first step tripling the response, a single noisy step (Di-8-ANEPPS ratios 1.38 to
4.04 by cell); with the field proportional to the volts
fig7b-charging-during-pulse fails.

How the analysis differs from the paper's: the cell outline is traced automatically on each set's baseline image
(three used sets traced again within a narrow window after the first tracing caught another structure; a variant
outline on the outer membrane leaves the Di-8-ANEPPS outcomes unchanged), with polar
angles from the cell centre, where the paper drew the trace by hand and resampled by perimeter length;
frames without a laser flash are left out; the field direction is found from the data; the eight Di-8-ANEPPS
escalation cells are used where the paper reports five, and the four FluoVolt cells with the nine-set sequence
where it reports four; the field strengths are matched to the volts in order.

What was found about the dataset: the files hold raw images only, with no cell outline, field direction, pixel size
or field strength in V/cm; the amplitude is in generator volts in a free-text description; every description gives
an electrode distance of 1.00 mm where the paper gives 1.6 mm; the dandiset description mentions only Di-8-ANEPPS
though a third of the sets are FluoVolt; frame 0 of nearly every set and a few frames more hold no laser flash and are
not flagged; the dandiset has eight Di-8-ANEPPS and five or six FluoVolt escalation cells where the paper reports five
and four, and five more Di-8-ANEPPS cells (2025-01-22) that the paper does not describe; and the Gd3+, 0.3-us,
two-pulse, alternating-amplitude and Ca2+ experiments are not in it.